# 🧹 3장 · 지저분한 데이터 정리하기: 판다스와 데이터 정제 (v4: 책 v5판 실습 노트북)

『빅데이터분석 with 파이썬』 3장의 **05. 파이썬 실습**과 **미니 프로젝트**를 그대로 따라 하는 노트북입니다. 실습 번호는 책과 같습니다.

| 표시 | 뜻 | 실습 |
|---|---|---|
| ★ 필수 | 수업 시간에 모두 함께 | 3-1 ~ 3-6 |
| ☆ 선택 | 자습이나 심화 | 3-7 ~ 3-10 |

**데이터**
- 따릉이: UCI "Seoul Bike Sharing Demand" 8,760시간(2017-12-01 ~ 2018-11-30). GitHub에 올라온 사본을 바로 읽습니다.
- 미니 프로젝트: 강의 자료 **지하철_원자료_표본.zip** (서울교통공사 "역별 일별 시간대별 승하차인원 정보" 원자료 여섯 파일에서 강남·신림·홍대입구역 줄만 그대로 옮긴 것). 미니 프로젝트 전에 왼쪽 📁 파일 창에 끌어다 놓으십시오.

실습마다 **단계(❶❷❸❹)별로 코드 셀이 나뉘어** 있고, 셀 위의 글은 책의 줄별 설명과 상자입니다. **[직접 바꿔 보기]** 셀은 숫자나 열 이름을 바꿔 가며 여러 번 돌려 보십시오.

코드 셀은 **위에서부터 차례로** 실행하십시오(`Shift + Enter`). 아래 셀은 위에서 만든 변수를 이어서 씁니다.

## 0 · 준비

그래프의 한글이 네모(□)로 깨지지 않게 해 주는 **koreanize-matplotlib**를 설치합니다. 판다스, 넘파이, 맷플롯립은 코랩에 이미 들어 있습니다.

In [ ]:
!pip install -q koreanize-matplotlib

**데이터 받기** (깃허브 저장소판에서 덧붙인 셀) — 아래 셀은 이 장이 쓰는 강의 자료 파일을 저장소의 `data` 폴더에서 코랩으로 내려받습니다.

In [ ]:
# 이 장의 강의 자료 파일을 책 깃허브 저장소에서 받아 옵니다.
# (파일을 코랩 왼쪽 📁 파일 창에 직접 끌어다 올렸다면 이 셀은 건너뛰어도 됩니다.)
import os, urllib.parse, urllib.request
BASE = "https://raw.githubusercontent.com/yeonu69/bigdata-python-book/main/ch03/data/"
for f in [
    "bike_daily_clean.csv",
    "bike_hourly_clean.csv",
    "cleaning_log.csv",
    "지하철_3개역_정제_2021_2024.csv",
    "지하철_원자료_표본.zip",
]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(BASE + urllib.parse.quote(f), f)
    print("준비됨:", f, f"({os.path.getsize(f) / 1e6:.1f} MB)")

## 05. 파이썬 실습: 정제된 하루 단위 표 만들기

실습은 코랩(Colab)에서 노트북 "3장_판다스와데이터정제_실습_colab_v4.ipynb"를 위에서부터 차례로 실행합니다. 한글 글꼴 설정은 노트북 첫 셀에 있고, 따릉이 데이터는 인터넷 주소에서 바로 읽습니다¹⁰⁾. 실행 결과는 판다스 3.0에서 돌린 것입니다. 판다스 2.x에서는 글자 열의 자료형이 `str` 대신 `object`로 보이지만 숫자는 같습니다.

> **[STEP] 실습 전에 미리 살펴보기**
>
> 데이터: 서울 따릉이 시간 기록 8,760행(실습 예제 3-4~3-13). 미니 프로젝트는 강의 자료의 지하철 원자료 표본입니다.
>
> 약속: 판다스 함수는 대부분 원래 표를 바꾸지 않고 새 표를 돌려줍니다. 그래서 `df = …`처럼 받아 둡니다. 원본 `raw`는 끝까지 고치지 않습니다.

**준비: 라이브러리 불러오기** (한글 글꼴 설정을 함께 합니다)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import koreanize_matplotlib            # 코랩: 한글 글꼴을 자동으로 설정
except ImportError:                        # 나눔고딕 파일이 이미 있는 컴퓨터
    import glob, os
    import matplotlib.font_manager as fm
    for f in glob.glob(os.path.expanduser("~/.fonts/Nanum*.ttf")):
        fm.fontManager.addfont(f)
    plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
pd.set_option("display.width", 100)       # 표를 한 줄에 넓게 찍기
pd.set_option("display.max_columns", 20)  # 열을 … 로 줄이지 않기

- `import 판다스 as pd`처럼 줄임 이름을 붙이는 것은 모두가 지키는 약속입니다. 인터넷에서 찾은 코드도 대부분 `pd.`, `np.`, `plt.`로 시작합니다.
- `set_option( )` 두 줄은 출력 모양만 정합니다. 열이 많은 표가 중간에서 줄바꿈되거나 가운데 열이 "…"로 줄지 않게 합니다. 계산 결과에는 영향이 없습니다.

### 먼저: 04절 [실습 예제 3-1~3-3]

책 04절의 [예제 3-1~3-3]에서 손으로 푼 숫자를 같은 코드로 확인합니다. 따릉이 파일은 아직 쓰지 않습니다.

#### [실습 예제 3-1 ★] [예제 3-1]을 파이썬 코드로 구현하기

[예제 3-1]에서 손으로 구한 평균과 중앙값을 판다스로 다시 구합니다. 손 계산과 같은 숫자가 나오는지 확인하십시오.

**❶ 키 다섯 개를 시리즈로 만들고 평균·중앙값 구하기**

In [ ]:
import pandas as pd
h = pd.Series([162, 168, 171, 174, 300])   # 300은 173의 오기
print("평균:", h.mean(), " 중앙값:", h.median())

- `pd.Series([…])`: 숫자 목록을 시리즈(이름표가 붙은 열 하나)로 만듭니다. 인덱스는 0부터 자동으로 붙습니다.
- `.mean( )`: 모두 더해 개수로 나눕니다(식 3-1). 975 ÷ 5 = 195입니다.
- `.median( )`: 값을 줄 세워 한가운데 값을 돌려줍니다. 다섯 개라 세 번째인 171입니다.

**❷ 300을 173으로 고치고 다시 구하기**

In [ ]:
h2 = h.replace(300, 173)                    # 원본 h는 그대로 두고 새 시리즈로
print("평균:", h2.mean(), " 중앙값:", h2.median())
print("평균이 움직인 거리:", round(h.mean() - h2.mean(), 1))

- `.replace(옛 값, 새 값)`: 같은 값을 모두 바꾼 **새** 시리즈를 돌려줍니다. 원본을 고치지 않으므로 고치기 전후를 견줄 수 있습니다.
- `round(값, 1)`: 소수 첫째 자리까지 반올림합니다. 빼기 결과에 붙는 아주 작은 오차(25.400000000000006 같은)를 감춥니다.

#### [실습 예제 3-2 ★] [예제 3-2]를 파이썬 코드로 구현하기

[예제 3-2]의 9일치 대여량으로 사분위수와 울타리를 구하고, 울타리 밖의 값을 골라낸 뒤 상자 그림으로 확인합니다.

**❶ 사분위수와 울타리 구하기**

In [ ]:
x = pd.Series([2, 3, 3, 4, 5, 5, 6, 7, 20])
q1, q3 = x.quantile(0.25), x.quantile(0.75)
iqr = q3 - q1                                # (식 3-2)
upper, lower = q3 + 1.5 * iqr, q1 - 1.5 * iqr
print("Q1:", q1, " Q3:", q3, " IQR:", iqr)
print("위 울타리:", upper, " 아래 울타리:", lower)

- `.quantile(0.25)`: 줄 세운 값의 (n − 1) × 0.25번째 자리 값입니다. 9개면 2번째(0번째부터 셈)라 3입니다. 자리가 칸 사이에 오면 두 값 사이를 직선으로 이어 읽습니다([잠깐만요]).
- `q1, q3 = …, …`: 오른쪽 두 값을 왼쪽 두 이름에 차례로 넣습니다. 한 줄에 둘을 함께 구할 때 씁니다.

**❷ 울타리 밖의 값 골라내기**

In [ ]:
out = x[(x > upper) | (x < lower)]           # 위로 넘거나, 아래로 넘거나
print(out)

- `x > upper`: 값마다 True/False가 든 시리즈를 만듭니다. 이것을 `x[ ]` 안에 넣으면 True인 행만 남습니다(불리언 인덱싱).
- `|`: "또는"입니다. 판다스에서는 `or` 대신 `|`, `and` 대신 `&`를 쓰고, 조건마다 괄호로 감쌉니다.
- 왼쪽의 8은 인덱스입니다. 0부터 세었으므로 아홉 번째 날이 후보라는 뜻입니다.

**❸ 상자 그림으로 확인하기**

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 2.4))
ax.boxplot(x, orientation="horizontal", whis=1.5)
ax.axvline(upper, color="crimson", ls="--", label=f"위 울타리 {upper}")
ax.set_xlabel("하루 대여량(대)"); ax.set_yticks([])
ax.legend(loc="upper center")
plt.show()

*위 그래프: [실습 예제 3-2]의 상자 그림. 상자의 양 끝이 Q1(3)과 Q3(6)이고, 수염은 울타리 안의 가장 먼 값(2와 7)까지 뻗는다. 울타리(점선) 밖의 20만 점으로 따로 찍힌다.*

- `ax.boxplot(x, orientation="horizontal", whis=1.5)`: 상자 그림을 가로로 그립니다. `whis=1.5`가 IQR 규칙의 1.5입니다. 울타리 밖의 값은 점으로 따로 찍힙니다.
- `ax.axvline(값)`: 그 자리에 세로선을 긋습니다. 울타리를 직접 그어 상자 그림의 점과 같은 자리인지 봅니다.
- `ax.set_yticks([ ])`: 세로축 눈금(상자 번호 1)을 지웁니다.

#### [실습 예제 3-3 ★] [예제 3-3]을 파이썬 코드로 구현하기

[예제 3-3]의 일곱 날을 NaN이 든 시리즈로 만들고, 평균으로 채우기 전과 후의 표준편차를 견줍니다. 풀이 표를 코드로 만들어 손 계산과 맞춰 봅니다.

**❶ 빈 이틀을 NaN으로 두고 구하기**

In [ ]:
import numpy as np
s = pd.Series([4, 6, 8, 10, 12, np.nan, np.nan])     # 단위: 백 대
print("평균:", s.mean(), " 표준편차:", round(s.std(), 2), " 값의 개수:", s.count(), "/", len(s))

- `np.nan`: 빈칸(결측)입니다. 판다스의 `mean( )`, `std( )`는 NaN을 빼고 계산하므로 다섯 값으로 구한 것과 같습니다.
- `.count( )`는 NaN이 아닌 값의 개수, `len( )`은 NaN을 포함한 전체 길이입니다. 둘의 차이가 결측의 개수입니다.

**❷ 풀이 표를 코드로 만들기**

In [ ]:
d = s.dropna()                      # 값이 있는 다섯 날
dev = d - d.mean()                  # 편차
tab = pd.DataFrame({"x": d, "x-mean": dev, "(x-mean)^2": dev ** 2})
print(tab.to_string(index=False))
print("제곱의 합:", (dev ** 2).sum(), " ÷ (n-1):", (dev ** 2).sum() / (len(d) - 1))

- `d - d.mean()`: 시리즈에서 숫자 하나를 빼면 값마다 뺍니다. (식 3-3)의 xᵢ − x̄를 다섯 개 한꺼번에 구한 것입니다.
- `dev ** 2`: 값마다 제곱합니다. 표의 셋째 열이 [예제 3-3] 풀이 표의 셋째 열과 같습니다.
- 마지막 10.0이 분산이고, 그 제곱근 √10 ≈ 3.16이 ❶의 표준편차입니다.

**❸ 평균으로 채우고 (식 3-4)로 확인하기**

In [ ]:
f = s.fillna(s.mean())              # 빈 이틀을 8로
print("평균:", f.mean(), " 표준편차:", round(f.std(), 2), " 값의 개수:", f.count())
print("(식 3-4):", round(s.std() * np.sqrt((5 - 1) / (7 - 1)), 2))

- `.fillna(값)`: NaN 자리를 그 값으로 채운 새 시리즈를 돌려줍니다. `s.mean( )`은 NaN을 뺀 평균 8입니다.
- 평균은 8 그대로인데 표준편차는 3.16에서 2.58로 줄었습니다. (식 3-4)의 √((n − 1) ÷ (N − 1))를 곱한 값과 같습니다.

**❹ 넘파이와 견주기**

In [ ]:
print("np.std:", round(np.std(d), 2), " np.std(ddof=1):", round(np.std(d, ddof=1), 2), " d.std():", round(d.std(), 2))

- `np.std(d)`는 n으로, `d.std( )`는 n − 1로 나눕니다. `ddof=1`을 주면 넘파이도 n − 1로 나눕니다([잠깐만] n으로 나눌까, n − 1로 나눌까).

### 이제 따릉이 파일로: [실습 예제 3-4~3-13]

#### [실습 예제 3-4 ★] 파일 읽기와 첫인상 살피기

따릉이 파일을 인코딩 없이 읽어 오류를 확인한 뒤 라틴-1로 다시 읽으시오. 열 이름을 짧게 바꾸고, 첫 점검 다섯 줄로 크기·값·자료형·범위·빈칸을 확인한 뒤 습도를 히스토그램으로 그리시오.

이 실습에서 쓰는 것: `read_csv(encoding=)`, `try/except`, `rename( )`, `shape`, `head( )`, `dtypes`, `describe( )`, `isna( ).sum( )`, `plot.hist( )`

**❶ 인코딩을 알려 주고 읽기**

In [ ]:
URL = ("https://raw.githubusercontent.com/syedsharin/"
       "Seoul-Bike-Sharing-Demand-Prediction/main/SeoulBikeData.csv")
try:
    pd.read_csv(URL)                        # 말하지 않으면 UTF-8로 읽는다
except UnicodeDecodeError as e:
    print("읽기 실패:", str(e)[:36])
raw = pd.read_csv(URL, encoding="latin-1")  # 원본: 끝까지 고치지 않는다
print(raw.shape)

- `URL = ("…" "…")`: 긴 글자를 괄호 안에서 두 줄로 나누어 적으면 파이썬이 하나로 이어 붙입니다.
- `pd.read_csv(URL)`: 인코딩을 말하지 않으면 UTF-8로 읽습니다. 이 파일의 열 이름 "Temperature(°C)"의 ° 글자가 UTF-8이 아니라서 오류가 납니다.
- `except UnicodeDecodeError as e:`: 그 오류가 나면 멈추지 않고 이 아래를 실행합니다. `e`에는 오류 메시지가 담기고, `str(e)[:36]`은 앞 36글자만 잘라 찍습니다.
- `encoding="latin-1"`: 바이트 하나를 글자 하나로 읽는 라틴-1로 다시 읽습니다. 결과는 `raw`라는 이름에 받아 두고 끝까지 고치지 않습니다.
- `raw.shape`: (행 수, 열 수)입니다. 8,760행은 365일 × 24시간과 같아야 합니다.


> **[코드 읽기] `try`와 `except`**
>
> `try:` 아래의 코드를 먼저 해 봅니다. 거기서 `except` 뒤에 적은 오류(여기서는 `UnicodeDecodeError`)가 나면 프로그램을 멈추지 않고 `except` 아래의 코드를 실행합니다. 오류 이름을 적지 않고 `except:`만 쓰면 어떤 오류든 삼켜 버려 진짜 문제를 놓치므로, 기다리는 오류의 이름을 꼭 적습니다. 노트북 맨 위의 한글 글꼴 설정 셀도 같은 방식으로 "없으면 설치"를 합니다.

> **[자주 하는 실수] 한글 파일이 깨지거나 읽히지 않을 때**
>
> `UnicodeDecodeError`가 나면 인코딩을 바꿔 봅니다. 공공데이터포털이나 엑셀에서 저장한 한글 CSV는 대부분 `encoding="cp949"`, 영어권 옛 파일은 `"latin-1"`, 최근 파일은 `"utf-8"`입니다. 오류 없이 읽혔는데 한글이 "Ã¬Â"처럼 보이면, 틀린 인코딩으로 읽은 것입니다. 라틴-1은 어떤 바이트도 오류 없이 읽기 때문에 한글 파일을 라틴-1로 읽으면 오류 대신 깨진 글자가 나옵니다. 읽은 뒤에는 반드시 `head( )`로 글자를 눈으로 확인합니다.


**❷ 열 이름을 짧게 바꾸기**

In [ ]:
names = {"Date": "date", "Rented Bike Count": "count", "Hour": "hour",
         "Temperature(°C)": "temp", "Humidity(%)": "humidity",
         "Wind speed (m/s)": "wind", "Visibility (10m)": "visibility",
         "Dew point temperature(°C)": "dew", "Solar Radiation (MJ/m2)": "solar",
         "Rainfall(mm)": "rain", "Snowfall (cm)": "snow", "Seasons": "season",
         "Holiday": "holiday", "Functioning Day": "functioning"}
df = raw.rename(columns=names)              # 고친 표는 새 이름 df에
print(list(df.columns))

- `names = {"옛 이름": "새 이름", …}`: 사전(dict)입니다. 괄호 안에서는 줄을 바꿔 적어도 한 줄로 읽습니다.
- `raw.rename(columns=names)`: 사전의 짝대로 열 이름을 바꾼 **새 표**를 돌려줍니다. `raw`는 그대로이고, 새 표는 `df`에 받습니다.
- 짧은 영어 소문자 이름은 `df["temp"]`처럼 쓰기 쉽고, 띄어쓰기·괄호·°가 없어 오타가 줄어듭니다.


**❸ 첫 점검 다섯 줄**

In [ ]:
print(df.shape)                                        # 1. 크기
print(df[["date", "hour", "count", "temp"]].head(3))   # 2. 값의 생김새
print(df.dtypes.astype(str).value_counts().to_dict())           # 3. 자료형
cols = ["count", "temp", "humidity", "dew", "rain"]
print(df[cols].describe().round(1).T[["mean", "min", "50%", "max"]])   # 4. 범위
print("빈칸:", df.isna().sum().sum())                  # 5. 빈칸

- `df[["date", "hour", …]]`: 대괄호 안에 **열 이름의 목록**을 넣으면 그 열들만 고른 표가 됩니다(괄호가 두 겹). `.head(3)`은 앞 세 행입니다.
- `df.dtypes`: 열마다 자료형입니다. `.astype(str)`로 이름을 글자로 바꾼 뒤 `.value_counts( )`로 "정수 열 몇 개, 실수 열 몇 개, 글자 열 몇 개"를 셉니다.
- `describe( )`: 숫자 열마다 개수·평균·표준편차·최솟값·사분위수·최댓값을 한 번에 구합니다. `.round(1)`은 소수 첫째 자리까지, `.T`는 표를 눕혀 열마다 한 줄로 보이게 하고, 마지막 `[[…]]`로 네 칸만 남깁니다.
- `df.isna( )`: 칸마다 비었으면 True인 표입니다. `.sum( )`을 한 번 하면 열마다, 두 번 하면 표 전체의 빈칸 수입니다.


> **[분석 기법] 처음 받은 데이터의 첫 점검 다섯 줄**
>
> 어떤 표를 받든 먼저 다섯 줄을 돌립니다. ① `df.shape`: 크기가 예상과 같은가(365일 × 24시간 = 8,760). ② `df.head( )`: 값이 어떻게 생겼나(날짜가 일/월/연인가). ③ `df.dtypes`: 숫자가 글자로 들어오지 않았나. ④ `df.describe( )`: 최솟값·최댓값이 말이 되나. ⑤ `df.isna( ).sum( )`: 빈칸이 몇 개인가.
>
> 쇼핑몰 주문 기록이라면 "가격 열이 글자(쉼표 때문)", "수량 최솟값 −1(반품)", "배송일 빈칸 300개(아직 배송 전)" 같은 문제가 이 다섯 줄에서 드러납니다. 글자 열은 `describe( )`에 나오지 않으므로 `df["열"].value_counts( )`로 따로 셉니다. min과 max는 한 줄씩 소리 내어 읽고 "이 값이 가능한가?"를 묻는 습관을 들이십시오.


> **[잠깐만] 열 고르기와 행 고르기: [ ], loc, iloc**
>
> `df["count"]`처럼 대괄호에 이름 하나를 넣으면 열 하나(시리즈), `df[["count", "temp"]]`처럼 목록을 넣으면 열 여러 개(데이터프레임)가 나옵니다. 행과 열을 함께 고를 때는 `loc`과 `iloc`을 씁니다. `df.loc[행 이름, 열 이름]`은 인덱스와 열의 **이름**으로, `df.iloc[행 번호, 열 번호]`는 0부터 세는 **위치**로 고릅니다.
>
> 예를 들어 `df.loc[0:2, ["date", "count"]]`는 인덱스 0~2(끝 포함)의 두 열, `df.iloc[0:2, 0:3]`은 앞 두 행(끝 미포함)의 앞 세 열입니다. `loc`은 끝을 포함하고 `iloc`은 넘파이처럼 끝을 빼는 것이 가장 헷갈리는 차이입니다. `df.loc[df["temp"] > 30, "count"]`처럼 행 자리에 조건을 넣으면 "기온 30도 넘는 시간의 대여량"이 됩니다(실습 예제 3-6의 조건 필터).


**❹ 습도를 히스토그램으로 그리기**

In [ ]:
df["humidity"].plot.hist(bins=50, color="#8EC5E8", edgecolor="white", figsize=(6, 3))
plt.xlabel("습도(%)"); plt.ylabel("시간 수"); plt.show()

- `df["humidity"]`: 열 하나를 꺼내면 시리즈입니다. 시리즈에 `.plot.hist( )`를 붙이면 바로 히스토그램이 그려집니다.
- `bins=50`: 값의 범위를 50칸으로 나눕니다. 칸이 너무 적으면 0%의 막대가 옆 칸과 섞여 보이지 않습니다. `edgecolor="white"`는 막대 사이에 흰 선을 그어 칸을 구분합니다.
- `figsize=(6, 3)`: 그림 크기(가로 6, 세로 3인치)입니다. `;`로 한 줄에 여러 명령을 이어 적었습니다.

*위 그래프: 습도 히스토그램. 10% 아래는 비어 있는데, 0%에만 막대가 따로 서 있다.*

**[직접 바꿔 보기]** 필요한 열만, 앞부분만 읽어 보시오. 큰 파일을 처음 열 때 쓰는 방법입니다.

In [ ]:
small = pd.read_csv(URL, encoding="latin-1", usecols=["Date", "Hour", "Rented Bike Count"], nrows=3)
print(small)

`usecols`는 읽을 열, `nrows`는 읽을 행 수입니다. 수 기가바이트 파일도 이렇게 앞부분만 열어 열 이름과 값 모양을 먼저 봅니다.

**[결과 읽기]** 파일은 8,760행 14열로 365일 × 24시간과 맞고, 빈칸은 하나도 없습니다. 그러나 "빈칸 0"이 "결측 없음"은 아닙니다. date 열의 자료형이 `str`, 곧 날짜처럼 생긴 글자이고(실습 예제 3-5에서 고침), min 칸에는 대여량 0과 습도 0이 보입니다. 그래프에서도 10% 아래가 비어 있는데 0%에만 막대가 따로 서 있습니다. "습도 0%가 가능한가?"라는 질문이 여기서 나옵니다(실습 예제 3-6).

#### [실습 예제 3-5 ★] 날짜를 바르게 읽기

날짜 글자를 날짜형으로 바꾸시오. 칸마다 짐작하게 했을 때와 형식을 알려 줬을 때를 견주고, 계절 열로 어느 쪽이 맞는지 확인한 뒤 달별 하루 평균이 어떻게 달라지는지 그리시오.

이 실습에서 쓰는 것: `pd.to_datetime(format=)`, `.dt.month`, `.dt.date`, `pd.crosstab( )`, `groupby( ).mean( )`, `plot.bar( )`

**❶ 두 가지 방법으로 읽기**

In [ ]:
wrong = pd.to_datetime(df["date"], format="mixed")      # 칸마다 따로 짐작
good = pd.to_datetime(df["date"], format="%d/%m/%Y")    # 형식을 알려 줌
print("짐작한 기간:", wrong.min().date(), "~", wrong.max().date())
print("알려 준 기간:", good.min().date(), "~", good.max().date())
print("자료형:", good.dtype)

- `pd.to_datetime(글자 열, format=…)`: 날짜 글자를 날짜형으로 바꿉니다. `format="mixed"`는 "칸마다 알아서 짐작하라"는 뜻이라, "01/12/2017"처럼 두 가지로 읽히는 칸을 월/일/연(1월 12일)으로 읽습니다.
- `format="%d/%m/%Y"`: `%d`(일) `/` `%m`(월) `/` `%Y`(네 자리 연도). 파일의 글자 모양과 **같은 순서, 같은 구분 기호**로 적습니다. 모양이 다른 칸이 하나라도 있으면 오류가 나므로, 형식이 섞이지 않았다는 확인도 됩니다.
- `.min( )`, `.max( )`: 가장 이른 날과 늦은 날입니다. `.date( )`는 시각(00:00:00)을 떼고 날짜만 보여 줍니다.
- `datetime64[us]`는 판다스의 날짜형입니다([us]는 마이크로초까지 담는다는 뜻). 이제 빼기(며칠 차이), 달 꺼내기, 정렬이 날짜 순서대로 됩니다.


> **[분석 기법] 날짜 형식 코드와 .dt**
>
> 형식 코드는 `%Y`(2018), `%y`(18), `%m`(03), `%d`(10), `%H`(14시), `%M`(05분), `%S`(초)를 글자 모양대로 이어 씁니다. "2018.03.10 14:05"라면 `format="%Y.%m.%d %H:%M"`, "20180310"이라면 `format="%Y%m%d"`입니다. 날짜형으로 바꾼 열에는 `.dt`를 붙여 `.dt.year`(연도), `.dt.month`(달), `.dt.day`(일), `.dt.dayofweek`(요일, 월요일 0), `.dt.hour`(시)를 바로 꺼냅니다.
>
> 주문일, 가입일, 진료일처럼 날짜가 있는 데이터는 모두 이 두 가지로 시작합니다. 두 날짜를 빼면 기간(`Timedelta`)이 나오므로 `(df["배송일"] - df["주문일"]).dt.days`로 "주문 뒤 며칠 만에 배송"을 바로 구합니다. 날짜로 바꿀 수 없는 칸이 섞인 파일은 `errors="coerce"`를 주면 그 칸만 NaT(날짜의 빈칸)가 되므로, `isna( ).sum( )`으로 몇 칸인지 세어 봅니다.


**❷ 다른 열과 맞춰 확인하기**

In [ ]:
print("다른 날로 옮겨 간 날:", (wrong != good).sum() // 24, "일")
print(pd.crosstab(wrong.dt.month, df["season"]).loc[[1, 7]] // 24)
print(pd.crosstab(good.dt.month, df["season"]).loc[[1, 7]] // 24)

- `wrong != good`: 두 날짜가 다른 행에서 True입니다. True는 1, False는 0으로 세므로 `.sum( )`이 다른 행 수이고, `// 24`(몫)로 날수로 바꿉니다.
- `pd.crosstab(가, 나)`: 가와 나의 조합마다 행 수를 세는 **교차표**입니다. 행은 짐작한 달, 열은 계절입니다.
- `.loc[[1, 7]]`: 교차표에서 1월과 7월 두 행만 고릅니다. 24로 나누어 시간 수를 날수로 바꿉니다.
- 같은 표를 `good`으로 한 번 더 만들어 견줍니다. 바르게 읽었다면 1월은 모두 겨울, 7월은 모두 여름이어야 합니다.


**❸ 달별 하루 평균 견주기**

In [ ]:
cmp = pd.DataFrame({"바르게": df.groupby(good.dt.month)["count"].mean() * 24,
                    "잘못": df.groupby(wrong.dt.month)["count"].mean() * 24}).round()
print(cmp.T.astype(int).to_string())
print("가장 많은 달 ÷ 가장 적은 달:", (cmp.max() / cmp.min()).round(1).to_dict())
df["date"] = good                           # 이제부터는 바르게 읽은 날짜

- `df.groupby(good.dt.month)["count"].mean( )`: 바르게 읽은 날짜의 달(1~12)로 행을 묶고, 묶음마다 대여량 평균을 냅니다. 묶는 기준은 df의 열이 아니어도 길이만 같으면 됩니다.
- `* 24`: 한 시간 평균에 24를 곱해 하루 평균으로 바꿉니다.
- `pd.DataFrame({"이름": 시리즈, …})`: 시리즈 두 개를 열로 나란히 붙인 표를 만듭니다. 인덱스(달)가 같은 행끼리 맞춰 붙습니다.
- `cmp.T.astype(int).to_string( )`: 표를 눕혀(달이 열) 정수로 바꾸고, 열을 줄이지 않고 모두 찍습니다.
- `cmp.max( ) / cmp.min( )`: 열마다 가장 많은 달을 가장 적은 달로 나눈 값입니다. 계절 차이의 크기입니다.
- `df["date"] = good`: 마지막 줄에서야 df의 date 열을 날짜형으로 바꿉니다. 확인을 마친 뒤에 고칩니다.


**❹ 막대그래프로 그리기**

In [ ]:
cmp.plot.bar(color=["#3F7CC0", "#F2A65A"], figsize=(7, 3))
plt.xlabel("달"); plt.ylabel("하루 평균 대여량(대)"); plt.xticks(rotation=0); plt.show()

- 데이터프레임에 `.plot.bar( )`를 붙이면 인덱스(달)가 가로축, 열 두 개가 나란한 막대가 됩니다. `color`는 열 순서대로 색을 줍니다.
- `plt.xticks(rotation=0)`: 가로축 글자를 눕히지 않고 바로 세웁니다.

*위 그래프: 달별 하루 평균 대여량. 바르게 읽은 날짜(파랑)와 칸마다 짐작한 날짜(주황).*

> **[자주 하는 실수] 오류가 없으면 맞게 읽은 줄 안다**
>
> `format`을 적지 않거나 `"mixed"`, `dayfirst=False`로 읽으면 판다스는 오류 없이 날짜를 만들어 냅니다. 13일 이후는 월이 될 수 없어 바르게 읽히고, 12일까지만 월과 일이 뒤바뀌므로 더 알아채기 어렵습니다. 날짜를 바꾼 뒤에는 ① 기간(`min`, `max`)이 파일 설명과 같은지, ② 날짜 수(`nunique( )`)가 맞는지, ③ 계절·요일처럼 날짜와 함께 움직이는 열이 맞는지 셋 중 둘은 꼭 확인합니다.

**[직접 바꿔 보기]** 날짜형 열에서 요일과 연-월을 꺼내 보시오.

In [ ]:
print(df["date"].dt.day_name().head(2).tolist(), df["date"].dt.dayofweek.head(2).tolist())
print(df["date"].dt.to_period("M").astype(str).value_counts().sort_index().head(3).to_dict())

`dt.day_name( )`은 요일 이름, `dt.dayofweek`는 요일 번호(월요일 0 ~ 일요일 6)입니다. `dt.to_period("M")`은 날짜를 "2017-12" 같은 달 단위로 바꾸어, 달마다 몇 행인지 세거나 묶을 때 씁니다. 12월은 31일 × 24시간 = 744행입니다.

**[결과 읽기]** 짐작한 날짜는 2017년 1월부터 2018년 12월까지 퍼져 있고, 132일이 다른 날로 옮겨 갔습니다. 짐작한 "1월"에는 겨울이 22일뿐이고 가을·봄·여름이 사흘씩 섞였지만, 형식을 알려 준 1월은 31일 모두 겨울입니다. 그 결과 계절 차이가 6.2배에서 2.5배로 줄었고, 그래프에서도 주황 막대가 납작합니다. 오류 메시지는 하나도 없었으니, 기간과 계절 열을 확인하는 ❷단계가 없었다면 지나쳤을 것입니다.

#### [실습 예제 3-6 ★] 숨은 결측 찾기: 운영 중단의 0과 습도 0%

운영하지 않은 시간의 규모와 달별 분포를 확인하시오. 습도 0%인 시간을 찾아 기온과 이슬점으로 계산한 습도와 견주고, 결측으로 바꾸시오.

이 실습에서 쓰는 것: 조건 필터 `df[조건]`, `nunique( )`, `groupby( ).size( )`, `np.exp( )`, `~`(아니다), `replace( )`, `isna( )`, `plt.scatter( )`

**❶ 운영하지 않은 시간 고르기**

In [ ]:
off = df[df["functioning"] == "No"]         # 조건이 True인 행만
print("운영 안 한 시간:", len(off), " 날짜 수:", off["date"].nunique())
print("대여량이 0인 시간:", (df["count"] == 0).sum())
print("운영 안 했는데 대여가 있던 시간:", ((df["functioning"] == "No") & (df["count"] > 0)).sum())
print("달마다:", off.groupby(off["date"].dt.month).size().to_dict())

- `df["functioning"] == "No"`: 행마다 True/False가 든 시리즈(불리언 마스크)를 만듭니다. 이것을 다시 `df[ ]`에 넣으면 True인 행만 남는 **조건 필터**입니다.
- `len(off)`: 고른 행 수, `off["date"].nunique( )`: 그 안의 서로 다른 날짜 수입니다.
- `(df["count"] == 0).sum( )`: 대여량이 0인 시간의 수입니다. 둘이 같다면 "0은 모두 운영 중단 때문"이라는 짐작이 맞습니다.
- `(조건1) & (조건2)`: 두 조건이 모두 True인 행입니다. 운영하지 않았다면서 대여가 있는 모순된 시간이 있는지 확인합니다.
- `off.groupby(달).size( )`: 달마다 행 수입니다. `size( )`는 값과 상관없이 행을 세고, `.to_dict( )`는 한 줄로 찍기 좋게 사전으로 바꿉니다.


> **[분석 기법] 조건 필터: 원하는 행만 고르기**
>
> 조건 필터는 두 걸음입니다. 먼저 `mask = df["열"] == 값`으로 행마다 True/False가 든 **불리언 마스크**를 만들고, `df[mask]`로 True인 행만 남깁니다. 여러 조건은 괄호로 묶어 `&`(그리고), `|`(또는)로 잇고, `~mask`는 반대입니다. 목록 안의 값인지는 `df["season"].isin(["Summer", "Autumn"])`, 범위 안인지는 `df["temp"].between(20, 30)`으로 묻습니다.
>
> 마스크는 더하면 개수(`mask.sum( )`), 평균 내면 비율(`mask.mean( )`)이 됩니다. 주문 기록에서 `orders[(orders["price"] <= 0) | (orders["qty"] < 0)]`로 말이 안 되는 주문만 골라 보는 식으로, 정제의 "의심하기"는 대부분 조건 필터로 합니다. 고른 행은 `len( )`으로 세고 `head( )`로 눈으로 봅니다.


**❷ 증인 열로 습도 다시 계산하기**

In [ ]:
zero = df["humidity"] == 0
calc = 100 * np.exp(17.625 * df["dew"] / (243.04 + df["dew"])) \
           / np.exp(17.625 * df["temp"] / (243.04 + df["temp"]))
print("습도 0%:", zero.sum(), "시간,", df.loc[zero, "date"].dt.date.nunique(), "일")
print("보통 시간의 차이 평균:", round((calc - df["humidity"])[~zero].abs().mean(), 2), "%p")
print("0%인 시간의 계산값:", round(calc[zero].min()), "~", round(calc[zero].max()), "%")

- `zero`: 습도가 0인 시간에 True인 마스크입니다. 여러 번 쓸 것이라 이름을 붙여 둡니다.
- `calc = …`: 기온과 이슬점으로 구한 습도입니다(3.4의 각주 7, 마그누스 근사식). `np.exp( )`는 넘파이 함수지만 판다스 열에 그대로 써서 8,760행을 한 번에 계산합니다. 줄 끝의 `\`는 "다음 줄에 이어진다"는 표시입니다.
- `df.loc[zero, "date"]`: 습도 0인 행의 날짜 열입니다. `.dt.date.nunique( )`로 며칠에 걸쳤는지 셉니다.
- `(calc - df["humidity"])[~zero]`: 계산값과 기록값의 차이 가운데 `~zero`(습도가 0이 **아닌**) 시간만 고릅니다. `.abs( ).mean( )`은 차이의 크기 평균입니다.
- 보통 시간에 차이가 작다면 이 식은 믿을 만한 "증인"이고, 0%인 시간에서 크게 어긋나면 기록이 틀린 쪽입니다.


**❸ 0%를 결측(NaN)으로 바꾸기**

In [ ]:
print("바꾸기 전 평균:", round(df["humidity"].mean(), 2), " 빈칸:", df["humidity"].isna().sum())
df["humidity"] = df["humidity"].replace(0, np.nan)   # 0% → NaN
print("바꾼 뒤 평균:", round(df["humidity"].mean(), 2), " 빈칸:", df["humidity"].isna().sum())
print("개수(count):", df["humidity"].count(), " 전체 행:", len(df))

- `replace(0, np.nan)`: 값 0을 모두 NaN(결측)으로 바꾼 새 열을 돌려주고, 같은 이름의 열에 다시 넣습니다.
- `isna( ).sum( )`: 이제 이 17시간이 "빈칸"으로 세어집니다.
- `mean( )`: 판다스는 평균을 낼 때 NaN을 빼고 계산합니다. 0이 17개 빠지니 평균이 조금 오릅니다.
- `count( )`: 빈칸이 아닌 값의 개수, `len(df)`: 빈칸을 포함한 전체 행 수입니다. 둘의 차이가 결측 수입니다.


> **[잠깐만] NaN: "여기 값이 없다"는 표시**
>
> NaN(Not a Number)은 판다스가 빈칸을 나타내는 값입니다. 파일의 빈칸, `None`, 날짜의 빈칸(NaT)은 읽을 때 모두 NaN 쪽으로 바뀝니다. NaN은 계산에서 저절로 빠집니다. `mean( )`, `sum( )`, `std( )`는 NaN을 건너뛰고, `count( )`는 NaN이 아닌 값만 셉니다. 그래서 0을 NaN으로 바꾸면 "0이었던 칸"이 평균에서 빠지는 것입니다.
>
> 한 가지 함정이 있습니다. NaN은 자기 자신과도 같지 않아서 `df["humidity"] == np.nan`은 모든 행이 False입니다. 빈칸은 반드시 `isna( )`로 찾고, 빈칸이 아닌 것은 `notna( )`로 찾습니다. 빈칸을 다루는 함수는 셋입니다. `isna( )`(찾기), `dropna( )`(빼기), `fillna(값)`(채우기, 실습 예제 3-10).


**❹ 기록과 계산을 산점도로 견주기**

In [ ]:
plt.figure(figsize=(5, 4))
plt.scatter(calc[~zero], raw.loc[~zero, "Humidity(%)"], s=3, color="#8EC5E8", label="보통 시간")
plt.scatter(calc[zero], raw.loc[zero, "Humidity(%)"], s=30, color="#E04B3C", label="습도 0%")
plt.plot([0, 100], [0, 100], color="gray", linewidth=1)          # 같으면 이 선 위
plt.xlabel("기온·이슬점으로 계산한 습도(%)"); plt.ylabel("기록된 습도(%)")
plt.legend(); plt.show()

- `plt.scatter(가로, 세로)`: 시간 하나를 점 하나로 찍습니다. 가로는 계산값, 세로는 기록값입니다. 원본 `raw`의 습도를 써서 0%가 그대로 보이게 했습니다.
- `s=`는 점 크기, `label=`은 범례에 쓸 이름입니다. 빨간 점(습도 0%)을 크게 찍어 눈에 띄게 합니다.
- `plt.plot([0, 100], [0, 100])`: (0, 0)에서 (100, 100)까지 대각선을 긋습니다. 계산과 기록이 같으면 점이 이 선 위에 놓입니다.

*위 그래프: 기록된 습도와 기온·이슬점으로 계산한 습도. 보통 시간은 대각선 위에 있고, 습도 0%인 17시간(빨강)만 바닥에 깔려 있다.*

> **[자주 하는 실수] and, or 대신 &, |와 괄호**
>
> 여러 조건을 이을 때 `df[df["temp"] > 30 and df["hour"] == 15]`라고 쓰면 "The truth value of a Series is ambiguous"라는 오류가 납니다. `and`, `or`, `not`은 True/False **하나**에 쓰는 말이고, 열 전체에는 `&`(그리고), `|`(또는), `~`(아니다)를 씁니다. 또 `&`가 `>`, `==`보다 먼저 계산되므로 조건마다 괄호를 칩니다. `df[(df["temp"] > 30) & (df["hour"] == 15)]`가 바른 모양입니다.

**[직접 바꿔 보기]** 숨은 결측이 다른 열에도 있는지, 말이 안 되는 값을 조건 필터로 찾아보시오.

In [ ]:
check = {"기온이 이슬점보다 낮음": (df["temp"] < df["dew"]).sum(),
         "음수 대여량": (df["count"] < 0).sum(),
         "시정 0": (df["visibility"] == 0).sum(),
         "비가 오는데 습도 30% 미만": ((df["rain"] > 0) & (df["humidity"] < 30)).sum()}
print({k: int(v) for k, v in check.items()})

"말이 안 되는 조합"을 조건으로 적어 세어 보는 것이 숨은 결측 찾기의 기본입니다. 기온이 이슬점보다 낮을 수는 없고(공기가 물을 더 품을 수 없으므로), 비가 오는데 습도가 아주 낮기도 어렵습니다. `int(v)`는 넘파이 정수를 보통 정수로 바꾸어 깔끔하게 찍습니다. 모두 0이니, 이 데이터의 숨은 결측은 습도 0%와 운영 중단의 0 두 가지라고 볼 수 있습니다.

**[결과 읽기]** 대여량이 0인 시간은 정확히 운영하지 않은 295시간이고, 운영하지 않았는데 대여가 있던 시간은 없으며, 9~11월에 247시간이 몰려 있습니다. 습도는 보통 시간에 계산과 기록이 0.24%p밖에 다르지 않은데, 0%인 17시간은 계산하면 33~72%입니다. 그래프에서도 빨간 점만 대각선에서 떨어져 바닥에 깔려 있습니다. 이 17시간을 NaN으로 바꾸자 평균이 조금 오르고, 이제 판다스가 결측으로 알아봅니다.

#### [실습 예제 3-7 ★] 이상치 후보 찾기: IQR 규칙

운영한 시간의 대여량에 IQR 규칙을 쓰고, 후보가 언제인지 확인하시오. 이어서 시각마다 따로 울타리를 쳐 보고, 강수량에도 써 보시오.

이 실습에서 쓰는 것: `.copy( )`, `quantile( )`, f-문자열, `value_counts( )`, `dt.dayofweek`, `groupby( ).quantile( )`, `unstack( )`, `map( )`, `boxplot( )`

**❶ 전체 기준 울타리 치기**

In [ ]:
h = df[df["functioning"] == "Yes"].copy()       # 운영한 시간만
q1, q3 = h["count"].quantile([0.25, 0.75])      # Q1, Q3을 한 번에
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr   # 아래·위 울타리 (식 3-2)
print(f"운영한 {len(h):,}시간  Q1 {q1:.0f}, 중앙값 {h['count'].median():.0f}, Q3 {q3:,.0f}")
print(f"IQR {iqr:.0f},  아래 울타리 {lower:,.0f},  위 울타리 {upper:,.0f}")

- `df[…].copy( )`: 운영한 시간만 골라 **따로 떼어 낸 새 표**를 만듭니다. 이 뒤에 h에 열을 더해도 df는 바뀌지 않습니다([자주 하는 실수] 참고).
- `quantile([0.25, 0.75])`: 25%와 75% 자리의 값, 곧 Q1과 Q3을 한 번에 돌려줍니다. 두 값을 `q1, q3 = …`으로 나누어 받습니다.
- `lower, upper = …`: [식 3-2]의 두 울타리입니다. 쉼표로 두 이름에 두 값을 한 줄에 넣습니다.
- 아래 울타리가 음수이면 대여량(0 이상)에는 아래쪽 후보가 없다는 뜻입니다.


> **[코드 읽기] f-문자열의 `:,.0f`**
>
> 따옴표 앞에 f를 붙이면 중괄호 안에 변수나 식을 넣을 수 있습니다. 중괄호 안의 `:` 뒤는 찍는 모양입니다. `{upper:,.0f}`의 `,`는 천 단위 쉼표, `.0f`는 "소수점 아래 0자리까지"라서 2389.0이 2,389로 찍힙니다. `{x:.2f}`는 소수 둘째 자리, `{r:.1%}`는 0.895를 89.5%로 찍습니다. 중괄호 안에서 열 이름을 쓸 때는 바깥과 다른 따옴표를 씁니다(`f"… {h['count'].median( )}"`).

> **[자주 하는 실수] 고른 표에 열을 더했더니 경고가 뜬다**
>
> `h = df[df["functioning"] == "Yes"]`처럼 `.copy( )` 없이 고른 뒤 `h["fence"] = …`로 열을 더하면 판다스 2.x에서는 SettingWithCopyWarning이 뜹니다. "h가 df의 일부를 비춘 것인지, 따로 떼어 낸 것인지 모르겠다. 원본이 바뀔 수도 있고 안 바뀔 수도 있다"는 경고입니다. 고른 표를 따로 고쳐 쓸 생각이면 처음부터 `.copy( )`를 붙여 떼어 냅니다. 반대로 원본의 일부만 고치고 싶다면 `df.loc[조건, "열"] = 값`처럼 원본에 직접 씁니다. 판다스 3.0부터는 고른 표가 늘 사본처럼 움직이지만, `.copy( )`를 적어 두면 어느 판에서나 뜻이 분명합니다.


**❷ 후보가 언제인지 보기**

In [ ]:
out = h[h["count"] > upper]
print("이상치 후보:", len(out), "시간")
print("많은 시각:", out["hour"].value_counts().head(3).to_dict())
print("계절:", out["season"].value_counts(normalize=True).round(2).to_dict())
print("그 가운데 평일:", round((out["date"].dt.dayofweek < 5).mean() * 100, 1), "%")
print(out.nlargest(3, "count")[["date", "hour", "count", "temp"]].to_string(index=False))

- `h[h["count"] > upper]`: 위 울타리보다 큰 시간만 고릅니다.
- `value_counts( )`: 값마다 몇 번 나왔는지 많은 순서로 셉니다. `.head(3)`으로 상위 세 시각만 봅니다. `normalize=True`를 주면 개수 대신 비율이 나옵니다.
- `(dt.dayofweek < 5)`: 월(0)~금(4)이면 True입니다. True/False에 `mean( )`을 쓰면 True의 비율이 됩니다.
- `nlargest(3, "count")`: count가 가장 큰 세 행입니다. 가장 튀는 값을 눈으로 확인합니다.


> **[분석 기법] value_counts: 글자 열과 후보를 세는 첫 도구**
>
> `df["season"].value_counts( )`는 값마다 개수를 많은 순서로 셉니다. `normalize=True`는 비율, `dropna=False`는 NaN도 하나의 값으로 셉니다. 설문의 "성별" 열에 "남", "남자", "M", "male"이 섞여 있거나, 빈칸이 "unknown"으로 적혀 있는 것도 이 한 줄에서 드러납니다. 숫자 열에 쓰면 같은 값이 몇 번 나왔는지 보이므로, 999처럼 유난히 자주 나오는 값(숨은 결측의 흔적)을 찾을 때도 씁니다. 두 열의 조합을 셀 때는 `pd.crosstab(가, 나)`(실습 예제 3-5)를 씁니다.


**❸ 시각마다 따로 울타리 치기, 강수량에 써 보기**

In [ ]:
q = h.groupby("hour")["count"].quantile([0.25, 0.75]).unstack()   # 시각 × (Q1, Q3)
fence = q[0.75] + 1.5 * (q[0.75] - q[0.25])                      # 시각마다 울타리
print(q.loc[[4, 8, 18]].round(0))
print(f"저녁 6시끼리: 위 울타리 {fence[18]:,.0f}")
own = h["hour"].map(fence)                                       # 행마다 자기 시각의 울타리
print("시각별로 잰 후보:", (h["count"] > own).sum(), "시간")
print("비가 0인 시간:", round((h["rain"] == 0).mean() * 100, 1), "%, 비 온 시간:", (h["rain"] > 0).sum())

- `h.groupby("hour")["count"].quantile([0.25, 0.75])`: 시각마다 Q1과 Q3을 구합니다. 결과는 (시각, 0.25)·(시각, 0.75) 두 겹의 인덱스를 가진 긴 시리즈입니다.
- `.unstack( )`: 안쪽 인덱스(0.25, 0.75)를 열로 펴서 시각(행) × Q1·Q3(열) 표로 만듭니다.
- `fence = q[0.75] + 1.5 * (…)`: 열끼리 계산하면 24개 시각의 울타리가 한 번에 나옵니다.
- `h["hour"].map(fence)`: 행마다 hour 값을 fence에서 찾아 그 울타리를 붙입니다. 8,465행에 시각별 울타리가 하나씩 붙은 시리즈 `own`이 생기고, `h["count"] > own`으로 행마다 자기 울타리와 견줍니다.
- 마지막 줄: 강수량은 대부분 0이라 Q1 = Q3 = 0, IQR = 0입니다. 그러면 비가 조금만 와도 모두 후보가 됩니다.


> **[분석 기법] 무엇과 견줄까: 집단별 기준**
>
> 같은 값도 견주는 무리에 따라 평범하기도 하고 튀기도 합니다. 반 전체로 보면 키 185cm는 튀지만 농구부 안에서는 평범합니다. 그래서 "시각별", "매장별", "요일별"처럼 비슷한 것끼리 묶어 기준을 따로 구합니다.
>
> 판다스에서는 두 가지 방법이 있습니다. ① 묶음마다 기준을 구한 뒤 `map( )`으로 행마다 붙이기: `fence = df.groupby("매장")["매출"].quantile(0.75)` → `df["매장"].map(fence)`. ② `transform( )`으로 한 번에 붙이기: `df.groupby("매장")["매출"].transform("median")`은 행마다 자기 매장의 중앙값을 돌려주므로 결과의 길이가 원래 표와 같습니다. 매장마다 매출 이상치를 찾거나, 학년마다 성적 울타리를 칠 때도 같은 코드입니다.


**❹ 시각별 상자 그림 그리기**

In [ ]:
h.boxplot(column="count", by="hour", figsize=(8, 3.5), grid=False)
plt.axhline(upper, color="#E04B3C", linestyle="--")      # 전체 기준 울타리
plt.title(""); plt.suptitle(""); plt.xlabel("시각"); plt.ylabel("한 시간 대여량(대)")
plt.show()

- `boxplot(column="count", by="hour")`: 시각마다 상자 하나씩, 24개를 나란히 그립니다. 상자는 Q1~Q3, 가운데 선은 중앙값, 수염 밖의 동그라미는 그 시각 기준의 후보입니다.
- `axhline(upper, …)`: 전체 기준 위 울타리 높이에 빨간 점선을 긋습니다.
- `title("")`, `suptitle("")`: 판다스가 자동으로 붙이는 제목("count grouped by hour")을 지웁니다.

*위 그래프: 시각별 대여량 상자 그림. 빨간 점선이 전체 기준 위 울타리(2,389대)다.*

**[직접 바꿔 보기]** 울타리의 1.5를 3으로 바꾸면 후보가 몇 시간으로 줄어드는지 보시오.

In [ ]:
for k in [1.5, 2, 3]:
    print(f"k = {k}:  위 울타리 {q3 + k * iqr:,.0f},  후보 {(h['count'] > q3 + k * iqr).sum()}시간")

k가 클수록 울타리가 멀어져 후보가 줄어듭니다. 1.5는 정해진 법칙이 아니라 "이쯤부터 한번 보자"는 관례입니다. 시뮬레이터 ③ 탭에서 k를 움직여 보면 같은 일이 눈에 보입니다.

**[결과 읽기]** 전체 기준으로는 152시간이 후보이고, 대부분 여름·가을 평일의 저녁 6~7시입니다. 퇴근길이라는 진짜 값입니다. 시각마다 따로 재면 저녁 6시의 울타리는 5,072대로 올라가고, 후보는 2시간으로 줄어듭니다. 강수량은 93.9%가 0이라 비 온 516시간이 모두 후보가 되므로, 이 규칙을 그대로 쓰면 안 되는 열입니다. IQR 규칙은 "버릴 값"이 아니라 "볼 값"을 고르는 도구입니다.

#### [실습 예제 3-8 ★] 하루 단위로 묶기

운영한 시간만 날짜별로 묶어 하루 단위 표를 만들고, 24시간이 다 있는 날만 남기시오. 0을 그대로 둔 365일과 하루 평균, 9월 평균, 바쁜 달 순서를 견주시오.

이 실습에서 쓰는 것: `groupby( ).agg( )`(이름 붙인 집계), 메서드 잇기, `reset_index( )`, `drop( )`, `sort_values( )`, 두 열로 묶기

**❶ 날짜별로 묶기**

In [ ]:
day = (h.groupby("date")
        .agg(count=("count", "sum"), temp=("temp", "mean"), rain=("rain", "sum"),
             snow=("snow", "sum"), humidity=("humidity", "mean"),
             season=("season", "first"), holiday=("holiday", "first"),
             hours=("hour", "size"))
        .reset_index())
print(day.shape)
print(day.head(2).to_string(index=False, float_format="%.1f"))

- 전체를 괄호로 감싸고 `.groupby( )`, `.agg( )`, `.reset_index( )`를 줄마다 하나씩 이었습니다. 함수의 결과에 바로 다음 함수를 붙이는 **메서드 잇기**입니다([코드 읽기] 참고).
- `h.groupby("date")`: 같은 날짜의 시간(최대 24행)끼리 묶습니다.
- `agg(새 열=("원래 열", "계산"), …)`: 새 열마다 무엇을 어떻게 계산할지 짝으로 적습니다. 대여량·비·눈은 하루 합(`sum`), 기온·습도는 하루 평균(`mean`), 계절·공휴일은 그날 첫 행의 값(`first`), hours는 그날의 행 수(`size`)입니다.
- 습도 평균은 실습 예제 3-6에서 NaN으로 바꾼 17시간을 빼고 냅니다.
- `reset_index( )`: 묶음 이름(date)이 인덱스로 올라가 있던 것을 보통 열로 되돌립니다. 그래야 다음 실습에서 `day["date"]`로 쓰고, 다른 표와 date로 합칠 수 있습니다.


> **[코드 읽기] 메서드 잇기와 괄호 줄바꿈**
>
> `a = df.groupby("date"); b = a.agg(…); day = b.reset_index( )`처럼 세 번 이름을 붙이는 대신, `day = (df.groupby("date").agg(…).reset_index( ))`로 이어 쓸 수 있습니다. 바깥을 괄호로 감싸면 점(`.`) 앞에서 마음대로 줄을 바꿀 수 있어, 한 줄에 한 걸음씩 위에서 아래로 읽힙니다. 중간 결과가 궁금하면 그 줄 뒤를 지우고 실행해 보면 됩니다. 너무 길게 이으면 어디서 틀렸는지 찾기 어려우니 다섯 걸음 안쪽이 좋습니다.

> **[분석 기법] groupby의 세 걸음: 나누고, 계산하고, 합친다**
>
> `groupby`는 (1) 같은 값끼리 행을 나누고(날짜별), (2) 묶음마다 계산하고(더하기·평균), (3) 결과를 한 표로 합칩니다. 이 순서를 "나누기-적용-합치기"(split-apply-combine)라고 부릅니다. 계산 자리에는 `sum`, `mean`, `median`, `min`, `max`, `size`(행 수), `count`(빈칸 아닌 값 수), `first`, `nunique`(서로 다른 값 수)를 씁니다.
>
> 주문 기록을 고객별로 묶어 `orders.groupby("고객").agg(횟수=("주문번호", "size"), 총액=("금액", "sum"))`으로 "고객마다 주문 횟수와 총액"을 구하거나, 센서 기록을 시간별로 묶어 "한 시간 평균"을 구하는 것도 같은 세 걸음입니다. 무엇을 더하고 무엇을 평균 낼지는 열마다 정해야 하는 결정입니다. 기온을 더하면 뜻이 없습니다.


**❷ 24시간이 다 있는 날만 남기기**

In [ ]:
print(day["hours"].value_counts().to_dict())
print(day.loc[day["hours"] < 24, ["date", "hours", "count"]].to_string(index=False))
print(len(day), "일 →", end=" ")
day = day[day["hours"] == 24].drop(columns="hours").reset_index(drop=True)
print(len(day), "일")

- `day["hours"].value_counts( )`: 하루에 몇 시간이 들었는지 셉니다. 24가 아닌 날이 몇 날인지 먼저 봅니다.
- `day.loc[조건, [열 목록]]`: 24시간이 안 되는 날의 세 열만 봅니다. 빼기 전에 무엇을 빼는지 눈으로 확인합니다.
- `print(…, end=" ")`: 줄을 바꾸지 않고 다음 출력을 같은 줄에 잇습니다. 바꾸기 전과 뒤의 행 수가 한 줄에 찍힙니다.
- `drop(columns="hours")`: 다 쓴 열을 뺍니다. `reset_index(drop=True)`는 빠진 행 때문에 듬성듬성해진 인덱스를 0부터 다시 매기고, 옛 인덱스는 열로 남기지 않습니다.


**❸ 0을 그대로 둔 365일과 견주기**

In [ ]:
all365 = df.groupby("date")["count"].sum()               # 0을 둔 365일
print(f"하루 평균  0 포함 {all365.mean():,.0f}대,  운영한 날만 {day['count'].mean():,.0f}대")
m0 = all365.groupby(all365.index.month).mean()
m1 = day.groupby(day["date"].dt.month)["count"].mean()
print(f"9월  0 포함 {m0[9]:,.0f}대,  운영한 날만 {m1[9]:,.0f}대")
print("바쁜 달 순서  0 포함:", m0.sort_values(ascending=False).index[:4].tolist(),
      " 운영한 날만:", m1.sort_values(ascending=False).index[:4].tolist())

- `all365`: 정제하지 않은 df를 날짜별로 더한 것입니다. 운영하지 않은 날은 0대로 들어 있습니다. 비교할 **기준선**입니다.
- `all365.index.month`: all365는 날짜가 인덱스인 시리즈라 `.dt` 없이 `.index.month`로 달을 꺼냅니다. 열일 때는 `day["date"].dt.month`처럼 `.dt`가 필요합니다.
- `sort_values(ascending=False)`: 큰 것부터 줄 세웁니다. `.index[:4].tolist( )`로 앞 네 달의 번호만 목록으로 꺼냅니다.


**❹ 막대그래프로 그리기**

In [ ]:
pd.DataFrame({"0 포함": m0, "운영한 날만": m1}).plot.bar(color=["#B9C0C8", "#E04B3C"], figsize=(7, 3))
plt.xlabel("달"); plt.ylabel("하루 평균 대여량(대)"); plt.xticks(rotation=0); plt.show()

- 두 시리즈를 열로 붙여 한 표로 만든 뒤 바로 막대그래프를 그립니다. 회색이 기준선, 빨강이 정제한 값입니다.

*위 그래프: 달별 하루 평균 대여량. 0을 그대로 둔 365일(회색)과 운영한 352일(빨강). 9~11월이 크게 달라진다.*

> **[자주 하는 실수] 모든 열을 sum( )으로 묶는다**
>
> `h.groupby("date").sum( )`처럼 열을 정하지 않고 묶으면 기온과 습도까지 더해져 "하루 기온 312도"가 나오고, 시각(hour)도 0+1+…+23 = 276이 됩니다. 판다스 3.0은 글자 열(계절)을 더할 수 없어 오류를 내기도 합니다. 묶을 때는 열마다 "더해야 뜻이 있는가(양), 평균이어야 하는가(상태), 하나만 고르면 되는가(이름표)"를 묻고 `agg( )`에 적습니다. 묶은 뒤에는 `reset_index( )`를 잊지 않았는지, 행 수가 날짜 수와 같은지 확인합니다.

**[직접 바꿔 보기]** 두 열로 묶어 계절마다 가장 바쁜 시각을 찾아보시오.

In [ ]:
by = h.groupby(["season", "hour"])["count"].mean().unstack("season")   # 시각 × 계절
print(by.idxmax().to_dict())
print(by.loc[[8, 18]].round(0).to_string())

`groupby(["season", "hour"])`처럼 목록을 주면 두 기준의 조합마다 묶고, `unstack("season")`은 계절을 열로 폅니다. `idxmax( )`는 열마다 가장 큰 값이 있는 인덱스(시각)입니다. 어느 계절이든 가장 바쁜 시각은 퇴근 시간인 저녁 6시이고, 겨울에는 모든 시각이 작습니다.

**[결과 읽기]** 8,465시간이 353일로 묶였고, 17시간만 운영한 10월 6일을 빼 352일이 남았습니다. 0을 빼자 하루 평균은 16,910대에서 17,489대로 오르고, 9월은 네 번째에서 두 번째로 바쁜 달이 되었습니다. 운영 중단이 9~11월에 몰려 있었기 때문입니다. 그래프에서도 회색과 빨강의 차이가 9~11월에만 큽니다.

#### [실습 예제 3-9 ★] 공휴일 표와 합치고 저장하기

공식 공휴일 표를 만들어 하루 단위 표와 합치고, 원래 Holiday 열과 어긋난 날을 찾으시오. 두 기준의 공휴일 효과를 견주고, 쉬는 날 열을 만들어 저장하시오.

이 실습에서 쓰는 것: `pd.DataFrame({…})`, `merge(on=, how=)`, `notna( )`, `!=`로 어긋난 행 찾기, `isin( )`, `to_csv( )`, `read_csv(parse_dates=)`

**❶ 공휴일 표를 만들어 날짜로 합치기**

In [ ]:
hol = pd.DataFrame({
    "date": ["2017-12-25", "2018-01-01", "2018-02-15", "2018-02-16", "2018-02-17",
             "2018-03-01", "2018-05-05", "2018-05-07", "2018-05-22", "2018-06-06",
             "2018-06-13", "2018-08-15", "2018-09-23", "2018-09-24", "2018-09-25",
             "2018-09-26", "2018-10-03", "2018-10-09"],
    "name": ["성탄절", "신정", "설날 연휴", "설날", "설날 연휴", "삼일절", "어린이날",
             "대체공휴일", "부처님오신날", "현충일", "지방선거일", "광복절", "추석 연휴",
             "추석", "추석 연휴", "대체공휴일", "개천절", "한글날"]})
hol["date"] = pd.to_datetime(hol["date"])        # 연-월-일 글자라 헷갈리지 않는다
print("공휴일 표:", hol.shape, " 겹친 날짜:", hol["date"].duplicated().sum())
n0 = len(day)
day = day.merge(hol, on="date", how="left")      # 날짜를 열쇠로 합치기
day["official"] = day["name"].notna()            # 공식 공휴일이면 True
print(n0, "행 →", len(day), "행,  공식 공휴일:", day["official"].sum(), "일")

- `pd.DataFrame({"열 이름": [값 목록], …})`: 목록 두 개로 18행 2열 표를 손으로 만듭니다. 두 목록의 길이가 같아야 합니다.
- `pd.to_datetime(hol["date"])`: "2018-05-05"처럼 연-월-일 순서의 글자는 형식을 적지 않아도 헷갈리지 않습니다. 합칠 열쇠는 **두 표에서 자료형이 같아야** 합니다(글자와 날짜는 같은 날이어도 짝이 맞지 않음).
- `hol["date"].duplicated( ).sum( )`: 공휴일 표에 같은 날짜가 두 번 있는지 합치기 **전에** 셉니다. 0이어야 합친 뒤 행이 불지 않습니다.
- `day.merge(hol, on="date", how="left")`: day의 행을 모두 남기고, 날짜가 같은 공휴일 이름을 옆 열에 붙입니다. 공휴일이 아닌 날의 name은 NaN입니다.
- `day["name"].notna( )`: 이름이 있으면 True, 곧 공식 공휴일 표시 열입니다.
- 합치기 전(`n0`)과 뒤의 행 수를 함께 찍습니다. 352행 그대로여야 합니다. 공휴일 18일 가운데 운영하지 않은 한글날(10월 9일)이 빠져 17일입니다.


> **[분석 기법] 두 표 합치기: merge의 how와 행 수 확인**
>
> `merge(다른 표, on="열쇠", how=…)`의 `how`는 열쇠가 한쪽에만 있는 행을 어떻게 할지 정합니다. `"left"`는 왼쪽 표의 행을 모두 남기고, `"inner"`는 두 표에 다 있는 열쇠만, `"outer"`는 어느 한쪽에라도 있으면 남깁니다. 주문 표에 고객 표를 붙일 때는 주문을 잃지 않도록 보통 `"left"`를 씁니다. 열쇠 이름이 두 표에서 다르면 `left_on="고객번호", right_on="id"`로 짝을 지어 줍니다.
>
> 합친 뒤에는 꼭 행 수를 셉니다. 고객 표에 같은 고객이 두 번 있으면 주문이 두 줄로 불어납니다. 겹침은 합치기 전에 `duplicated(subset=["열쇠"])`로 찾고, `drop_duplicates(subset=["열쇠"], keep="last")`로 나중 것만 남깁니다(2.1의 두 번 낸 설문, 미니 프로젝트 3단계). `merge(…, validate="many_to_one")`이라고 적어 두면 이런 겹침이 있을 때 판다스가 오류로 알려 줍니다. 위아래로 이어 붙일 때(같은 열의 표 여러 개)는 `merge`가 아니라 `pd.concat([표1, 표2])`를 씁니다.


**❷ 어긋난 날과 두 기준의 평균**

In [ ]:
diff = day[(day["holiday"] == "Holiday") != day["official"]]   # 두 표가 다른 날
print(diff[["date", "holiday", "name", "count"]].to_string(index=False))
for label, is_hol in [("원래 열", day["holiday"] == "Holiday"), ("공식 표", day["official"])]:
    print(f"{label} 기준  공휴일 {day.loc[is_hol, 'count'].mean():,.0f}대,"
          f"  그 밖의 날 {day.loc[~is_hol, 'count'].mean():,.0f}대")

- `(day["holiday"] == "Holiday") != day["official"]`: True/False 두 열을 `!=`로 견주면 한쪽만 True인 날, 곧 두 표가 다르게 말하는 날만 True입니다.
- `to_string(index=False)`: 행 번호 없이 표를 찍습니다.
- `for label, is_hol in [(이름, 마스크), …]`: 이름과 마스크의 짝 두 개를 차례로 돌며 같은 계산을 합니다. 같은 코드를 두 번 적지 않는 방법입니다.
- `day.loc[is_hol, "count"].mean( )`은 공휴일의, `day.loc[~is_hol, "count"].mean( )`은 그 밖의 날의 하루 평균입니다.


**❸ 쉬는 날 열을 만들고 저장하기**

In [ ]:
day["offday"] = (day["date"].dt.dayofweek >= 5) | day["official"]   # 주말이거나 공휴일
h["offday"] = (h["date"].dt.dayofweek >= 5) | h["date"].isin(hol["date"])
print(day.groupby("offday")["count"].agg(["size", "mean"]).round())
day.to_csv("bike_daily_clean.csv", index=False, encoding="utf-8-sig")
h.to_csv("bike_hourly_clean.csv", index=False, encoding="utf-8-sig")
check = pd.read_csv("bike_daily_clean.csv", parse_dates=["date"])
print("다시 읽은 표:", check.shape, check["date"].min().date(), "~", check["date"].max().date())
print(check.dtypes[["date", "count", "official", "offday"]].astype(str).to_dict())

- `dt.dayofweek >= 5`: 토(5)·일(6)요일이면 True입니다. `|`로 공식 공휴일과 이어 "쉬는 날" 열을 만듭니다.
- 시간 표 h에는 name 열이 없으므로 `h["date"].isin(hol["date"])`로 "그날이 공휴일 목록 안에 있는가"를 바로 묻습니다.
- `agg(["size", "mean"])`: 이름을 붙이지 않고 계산 이름만 목록으로 주면, 열 이름이 size, mean인 표가 나옵니다.
- `to_csv(파일 이름, index=False, encoding="utf-8-sig")`: 인덱스(0, 1, 2 …)는 저장하지 않고, 엑셀에서 열어도 한글이 깨지지 않게 저장합니다.
- `read_csv(…, parse_dates=["date"])`: CSV에는 자료형이 저장되지 않으므로, 다시 읽을 때 날짜 열을 날짜형으로 읽으라고 알려 줍니다. True/False 열은 저절로 bool로 읽힙니다.
- 저장한 파일을 다시 읽어 크기·기간·자료형을 확인하는 것까지가 저장입니다.


> **[분석 기법] 정제한 표를 저장하고 다시 읽기**
>
> CSV는 글자만 저장하므로 자료형이 사라집니다. 그래서 다시 읽을 때 `parse_dates=["date"]`로 날짜를, `dtype={"우편번호": str}`로 0으로 시작하는 번호(01234)를 지켜 줍니다. 저장할 때 `index=False`를 빼면 "Unnamed: 0"이라는 쓸모없는 열이 생깁니다. 한글이 든 표는 `encoding="utf-8-sig"`로 저장하면 엑셀과 코랩 어디서나 깨지지 않습니다.
>
> 자료형까지 그대로 지키고 싶으면 `df.to_parquet("파일.parquet")`, `pd.read_parquet( )`(파케이 형식)를 씁니다. 크기도 CSV보다 훨씬 작습니다. 다만 엑셀로 열 수 없어, 사람과 나눌 파일은 CSV, 내가 다시 쓸 중간 파일은 파케이로 나누어 쓰기도 합니다. 어느 쪽이든 저장한 뒤에는 다시 읽어 `shape`와 기간을 확인합니다.


**❹ 아침 8시를 그래프로 견주기**

In [ ]:
eight = h[h["hour"] == 8].groupby("offday")["count"].mean()
print(eight.round().to_dict())
eight.rename({False: "평일", True: "쉬는 날"}).plot.bar(color=["#3F7CC0", "#E04B3C"], figsize=(4, 3))
plt.xlabel(""); plt.ylabel("아침 8시 평균 대여량(대)"); plt.xticks(rotation=0); plt.show()

- 아침 8시 행만 골라 쉬는 날 여부로 묶고 평균을 냅니다.
- `rename({False: "평일", True: "쉬는 날"})`: 인덱스 값 False/True를 읽기 좋은 이름으로 바꿉니다(열 이름이 아니라 인덱스를 바꿀 때도 같은 함수).


*위 그래프: 아침 8시 평균 대여량. 평일과 쉬는 날이 세 배 넘게 다르다.*

**[직접 바꿔 보기]** 공휴일 표에 같은 날짜가 두 번 들어가면 어떻게 되는지 보시오.

In [ ]:
hol2 = pd.concat([hol, hol.iloc[[0]]])                  # 성탄절을 한 줄 더
test = day[["date", "count"]].merge(hol2, on="date", how="left")
print("합친 뒤 행 수:", len(test), "(352여야 함)")
try:
    day[["date", "count"]].merge(hol2, on="date", how="left", validate="many_to_one")
except pd.errors.MergeError as e:
    print("validate가 막음:", str(e)[:40])
print("how별 행 수:", {k: len(day[["date"]].merge(hol, on="date", how=k)) for k in ["left", "inner", "outer"]})

`pd.concat([표1, 표2])`는 두 표를 위아래로 잇습니다. 겹친 성탄절 때문에 그날이 두 줄이 되어 353행이 됩니다. `validate="many_to_one"`("왼쪽은 여러 번, 오른쪽은 한 번")을 적어 두면 판다스가 오류로 막아 줍니다. `how="inner"`는 두 표에 다 있는 17일만, `"outer"`는 어느 한쪽에라도 있는 353일(운영하지 않아 day에 없는 한글날 포함)을 남깁니다.

**[결과 읽기]** 합친 뒤에도 352행 그대로입니다. 두 표가 어긋난 날은 8일이고, 원래 열 기준으로는 공휴일에 5,000대 덜 타지만 공식 표 기준으로는 차이가 200대 안쪽입니다. 하루 합계는 평일(237일)과 쉬는 날(115일)이 비슷해도, 아침 8시는 평일이 세 배 넘게 많습니다. 4장과 5장은 이렇게 저장한 bike_daily_clean.csv와 bike_hourly_clean.csv에서 출발합니다.

#### [실습 예제 3-10 ☆] 결측을 채우면 무엇이 바뀌나

운영하지 않은 13일을 빈칸으로 두고, 지우기, 0으로 두기, 평균으로 채우기, 중앙값으로 채우기를 견주시오. 판다스와 넘파이의 표준편차도 견주시오.

이 실습에서 쓰는 것: `where( )`, `dropna( )`, `fillna( )`, 사전 내포, `std(ddof=)`, `interpolate( )`

**❶ 운영하지 않은 날을 빈칸으로 만들기**

In [ ]:
s = all365.where(~all365.index.isin(off["date"]))        # 운영 안 한 13일 → NaN
print("날수:", len(s), " 빈칸:", s.isna().sum(), " 값이 있는 날:", s.count())

- `all365`: 실습 예제 3-8에서 만든 365일의 하루 합계(날짜가 인덱스)입니다.
- `all365.index.isin(off["date"])`: 날짜마다 "운영하지 않은 날 목록에 있는가"를 묻는 마스크입니다. 앞의 `~`로 뒤집어 "운영한 날"에 True가 됩니다.
- `where(조건)`: 조건이 True인 곳은 그대로 두고, False인 곳을 NaN으로 바꿉니다. 조건 필터와 달리 행을 지우지 않고 자리를 남깁니다.
- off에는 7시간 멈춘 10월 6일도 들어 있어 그날의 합계(17시간분)도 빈칸이 됩니다. 그래서 빈칸은 13일이고, 남는 352일은 실습 예제 3-8의 day와 같은 날들입니다.


**❷ 네 가지 방법을 한 표로 견주기**

In [ ]:
ways = {"지우기": s.dropna(), "0으로": s.fillna(0),
        "평균으로": s.fillna(s.mean()), "중앙값으로": s.fillna(s.median())}
table = pd.DataFrame({k: [len(v), v.mean(), v.median(), v.std()] for k, v in ways.items()},
                     index=["날수", "평균", "중앙값", "표준편차"]).round(0)
print(table)

- `ways = {"이름": 결과, …}`: 방법마다 결과 시리즈를 사전에 담습니다. `dropna( )`는 NaN인 날을 빼고, `fillna(값)`은 NaN을 그 값으로 채웁니다.
- `s.mean( )`, `s.median( )`: NaN을 뺀 352일의 평균과 중앙값입니다. 이 값으로 빈칸 13개를 모두 똑같이 채웁니다.
- `{k: [ … ] for k, v in ways.items( )}`: **사전 내포**입니다. 방법마다 [날수, 평균, 중앙값, 표준편차] 네 값의 목록을 만들어 열로 씁니다.
- `index=[…]`: 네 값에 행 이름을 붙입니다. 이 표가 [표 3-6]입니다.


**❸ 9~11월을 그래프로 견주기**

In [ ]:
import matplotlib.dates as mdates
part = slice("2018-09-01", "2018-11-30")
plt.figure(figsize=(8, 3))
plt.plot(ways["0으로"][part], color="#B9C0C8", label="0으로")
plt.plot(ways["평균으로"][part], color="#3F7CC0", label="평균으로")
plt.scatter(s[part][s[part].isna()].index, [0] * s[part].isna().sum(), color="#E04B3C", s=20, label="빈칸이었던 날")
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))   # 눈금을 "월-일"로
plt.ylim(-1500, 40000); plt.ylabel("하루 대여량(대)"); plt.legend(ncol=3); plt.show()

- `slice("2018-09-01", "2018-11-30")`: 날짜 인덱스에서 이 기간만 자르는 표시입니다. `시리즈[part]`로 씁니다.
- `plt.plot( )`: 날짜 순서대로 선을 긋습니다. 0으로 둔 선은 빈칸인 날마다 바닥까지 꺼지고, 평균으로 채운 선은 그날 평균 높이에서 평평해집니다.
- `plt.scatter(빈칸인 날짜, [0, 0, …])`: 빈칸이었던 날을 바닥에 빨간 점으로 표시합니다.
- `mdates.DateFormatter("%m-%d")`: 날짜 눈금을 "09-15"처럼 짧게 찍어 글자가 겹치지 않게 합니다. `ylim`은 세로축 범위로, 위에 범례 자리를 남깁니다.

*위 그래프: 9~11월 하루 대여량. 0으로 둔 선(회색)은 빈칸인 날마다 바닥으로 꺼지고, 평균으로 채운 선(파랑)은 그날 평균 높이에서 평평하다.*

**❹ 판다스와 넘파이의 표준편차**

In [ ]:
x = pd.Series([4, 6, 8, 10, 12])                       # [예제 3-3]의 다섯 날
print("판다스 std( ):", round(x.std(), 2), "  넘파이 np.std( ):", round(np.std(x), 2))
print("ddof를 맞추면:", round(x.std(ddof=0), 2), round(np.std(x, ddof=1), 2))

- 판다스 `std( )`는 n − 1로, 넘파이 `np.std( )`는 n으로 나누는 것이 기본입니다([잠깐만] n으로 나눌까, n − 1로 나눌까).
- `ddof`(자유도 빼기)는 나누는 수를 n − ddof로 정합니다. `ddof=0`이면 n, `ddof=1`이면 n − 1입니다. 같은 값이 나와야 할 때는 둘을 맞춥니다.


**[직접 바꿔 보기]** 앞뒤 날을 잇는 선형 보간으로 채우고 평균·표준편차를 견주시오.

In [ ]:
lin = s.interpolate()                                   # 앞뒤 값 사이를 직선으로
print("보간:", round(lin.mean()), round(lin.std()), "  평균으로:", round(ways["평균으로"].mean()), round(ways["평균으로"].std()))
print(pd.DataFrame({"원래": s, "보간": lin, "전날 값": s.shift(1)}).loc["2018-09-17":"2018-09-20"].round(0))

`interpolate( )`는 빈칸의 앞뒤 값을 직선으로 이어 채웁니다. 날짜 순서가 있는 데이터에서는 "그 무렵의 값"으로 채우므로 표준편차를 덜 줄입니다. 9월 18·19일처럼 이틀 연달아 빈 곳은 17일과 20일 사이를 3등분한 값으로 채웁니다. `shift(1)`은 한 칸 아래로 밀어 "전날 값"을 옆에 놓습니다(12장 시계열에서 다시 씁니다).

**[결과 읽기]** 0으로 두면 평균과 중앙값이 내려가고 표준편차는 커집니다. 평균이나 중앙값으로 채우면 날수는 365일로 돌아오지만 표준편차가 줄어, 실제보다 고른 데이터처럼 보입니다. 그래프에서도 평균으로 채운 날은 앞뒤와 상관없이 평평합니다. 어느 방법이든 "무엇이 바뀌는가"를 알고 고르는 것이 중요하고, 이 장에서 하루 분석은 지우기를 골랐습니다.

#### [실습 예제 3-11 ☆] 정제 일지를 표로 저장하기

[표 3-5]의 정제 일지를 데이터프레임으로 만들어 CSV로 저장하고, 다시 읽어 새 결정을 한 줄 더하시오.

이 실습에서 쓰는 것: 튜플 목록으로 표 만들기, `to_csv( )`, `pd.concat( )`

**❶ 일지를 표로 만들어 저장하기**

In [ ]:
log = pd.DataFrame([
    ("인코딩", "latin-1로 읽음", "°C 바이트(0xB0)가 UTF-8이 아님", "14열"),
    ("날짜", "format='%d/%m/%Y'", "일/월/연 순서, 계절 열로 확인", "365일"),
    ("습도 0%", "결측(NaN)으로", "이슬점과 모순", "17시간"),
    ("운영 안 한 시간", "하루 분석에서 뺌", "0은 수요가 아니라 운영 중단", "295시간"),
    ("24시간 미만인 날", "뺌", "10월 6일은 17시간만 운영", "1일"),
    ("이상치 후보", "그대로 둠", "퇴근 시간의 진짜 값", "152시간"),
    ("공휴일", "공식 표를 합쳐 새 열로", "원래 열과 8일이 다름, 원래 열도 남김", "17일")],
    columns=["무엇을", "어떻게", "왜", "몇 건"])
log.to_csv("cleaning_log.csv", index=False, encoding="utf-8-sig")
print(len(log), "줄 저장")

- `pd.DataFrame([(…), (…), …], columns=[…])`: 튜플 하나가 한 행입니다. 행마다 네 값을 같은 순서로 적고, `columns`로 열 이름을 붙입니다. 결정을 하나 내릴 때마다 한 줄씩 늘려 가기 좋은 모양입니다.
- "몇 건"에는 그 결정이 건드린 행 수를 적습니다. 숫자는 위 실습의 출력에서 옮깁니다.


**❷ 다시 읽고 새 결정 한 줄 더하기**

In [ ]:
log = pd.read_csv("cleaning_log.csv")
new = pd.DataFrame([("습도 빈칸 17시간", "계산한 습도로 채움, 표시 열", "증인 열이 믿을 만함(차이 0.24%p)", "17시간")],
                   columns=log.columns)
log = pd.concat([log, new], ignore_index=True)
print(log[["무엇을", "어떻게", "몇 건"]].tail(3).to_string(index=False))

- `columns=log.columns`: 새 줄의 열 이름을 일지와 똑같이 맞춥니다. 열 이름이 다르면 `concat` 뒤에 열이 따로 생깁니다.
- `pd.concat([log, new], ignore_index=True)`: 일지 아래에 새 줄을 잇고, 행 번호를 0부터 다시 매깁니다. 실습 예제 3-13의 결정입니다.


정제한 표와 일지를 함께 저장해 두면, 석 달 뒤의 나도 왜 10월 6일을 뺐는지 알 수 있습니다. 일지는 코드보다 짧아 동료가 먼저 읽는 문서이기도 합니다.

#### [실습 예제 3-12 ☆] 넓은 표와 깔끔한 표 오가기

따릉이 시간 기록을 날짜 × 시각의 넓은 표로 바꾸어 한눈에 그려 보고, 다시 깔끔한 표로 되돌리시오. 두 기준으로 묶은 요약표도 만드시오.

이 실습에서 쓰는 것: `pivot( )`, `plt.imshow( )`, `melt( )`, `equals( )`, `pivot_table( )`

**❶ 깔끔한 표를 넓은 표로**

In [ ]:
wide = df.pivot(index="date", columns="hour", values="count")   # 날짜 × 시각
print("넓은 표:", wide.shape)
print(wide.iloc[:2, [0, 8, 18]])

- `pivot(index=행이 될 열, columns=열이 될 열, values=칸에 들어갈 열)`: 8,760행의 긴 표를 날짜 365행 × 시각 24열로 폅니다. 한 행이 하루, 한 칸이 그 시각의 대여량입니다.
- 날짜·시각 조합이 겹치면 칸 하나에 값 두 개가 들어가야 하므로 오류가 납니다. 오류가 없었다는 것도 "겹친 기록이 없다"는 확인입니다.
- `wide.iloc[:2, [0, 8, 18]]`: 앞 두 날의 0시·8시·18시 칸만 위치로 고릅니다.


> **[잠깐만] 넓은 표와 긴(깔끔한) 표, 언제 무엇을 쓸까**
>
> 같은 내용을 두 모양으로 적을 수 있습니다. **넓은 표**는 "날짜 × 시각"처럼 한 변수(시각)의 값이 열 이름이 된 표로, 사람이 눈으로 보거나 엑셀로 나누어 줄 때 편합니다. **긴 표**는 "날짜, 시각, 대여량" 세 열에 한 행이 관측 하나인 깔끔한 표로, `groupby`, `merge`, 그래프 함수가 모두 이 모양을 기대합니다. 그래서 받은 넓은 표는 먼저 `melt( )`로 길게 바꾸어 분석하고, 보고서에 넣을 표만 `pivot( )`이나 `pivot_table( )`로 다시 넓게 폅니다. 설문의 "문항1, 문항2, …" 열, 월별 매출의 "1월, 2월, …" 열도 모두 넓은 표입니다.


**❷ 넓은 표를 그림 한 장으로**

In [ ]:
plt.figure(figsize=(8, 3))
plt.imshow(wide.T, aspect="auto", cmap="Blues", interpolation="nearest")
plt.yticks([0, 6, 12, 18, 23]); plt.ylabel("시각")
plt.xticks(range(0, 365, 61), [d.strftime("%y-%m") for d in wide.index[::61]]); plt.xlabel("날짜")
plt.colorbar(label="대여량(대)"); plt.show()

- `plt.imshow(표)`: 표의 칸 하나를 색칠한 네모 하나로 그립니다. `wide.T`로 눕혀 가로가 날짜, 세로가 시각이 되게 했습니다. `cmap="Blues"`는 많을수록 진한 파랑입니다.
- `aspect="auto"`: 칸이 정사각형이 아니어도 그림 틀에 맞게 늘립니다.
- `xticks(위치, 글자)`: 61일마다 눈금을 두고, `strftime("%y-%m")`으로 날짜를 "18-03" 모양 글자로 바꾸어 붙입니다.
- 넓은 표는 계산에는 불편해도, 이렇게 한눈에 보는 그림에는 알맞습니다.

*위 그래프: 날짜 × 시각 대여량. 아침 8시와 저녁 6시의 가로 띠, 여름의 진한 덩어리, 운영하지 않은 날의 흰 세로줄이 한 장에 보인다.*

**❸ 넓은 표를 다시 깔끔한 표로**

In [ ]:
long = wide.reset_index().melt(id_vars="date", var_name="hour", value_name="count")
print("깔끔한 표:", long.shape)
back = long.sort_values(["date", "hour"]).reset_index(drop=True)
print("원래 표와 같은가:", back["count"].equals(df["count"]))

- `wide.reset_index( )`: 인덱스였던 date를 보통 열로 내립니다. `melt`가 기준 열로 쓰기 위해서입니다.
- `melt(id_vars="date", var_name="hour", value_name="count")`: date는 그대로 두고, 나머지 24개 열 이름을 hour 열의 값으로, 칸의 값을 count 열로 세웁니다. 365 × 24 = 8,760행이 됩니다.
- `melt`는 열 순서대로(0시의 365일, 1시의 365일 …) 행을 쌓으므로, 날짜·시각 순서로 다시 정렬합니다.
- `equals( )`: 두 열의 값이 순서까지 모두 같으면 True입니다. 펴고 접어도 잃은 값이 없다는 확인입니다.


**❹ 두 기준으로 묶은 요약표: pivot_table**

In [ ]:
summary = h.pivot_table(index="season", columns="offday", values="count", aggfunc="mean").round(0)
summary.columns = ["평일", "쉬는 날"]
print(summary.loc[["Spring", "Summer", "Autumn", "Winter"]])

- `pivot_table( )`은 `pivot( )`과 달리 같은 칸에 값이 여럿이면 `aggfunc`(여기서는 평균)으로 하나로 줄입니다. 계절 × 쉬는 날 여부마다 한 시간 평균 대여량을 냅니다.
- `summary.columns = […]`: 열 이름 False/True를 읽기 좋게 바꿉니다. `.loc[[…]]`로 계절 순서를 봄·여름·가을·겨울로 맞춥니다.


> **[분석 기법] pivot_table: 엑셀의 피벗 테이블을 한 줄로**
>
> `df.pivot_table(index="행 기준", columns="열 기준", values="값 열", aggfunc="mean")`은 엑셀의 피벗 테이블과 같습니다. `aggfunc`에는 `"sum"`, `"count"`, `"median"`이나 목록 `["mean", "size"]`을 줄 수 있고, `margins=True`를 주면 맨 아래·맨 오른쪽에 전체 합계 줄이 붙습니다. 같은 일을 `df.groupby(["행 기준", "열 기준"])["값 열"].mean( ).unstack( )`으로도 할 수 있습니다. 쇼핑몰이라면 `orders.pivot_table(index="지역", columns="요일", values="금액", aggfunc="sum")`이 "지역 × 요일 매출표"입니다. `pivot( )`은 칸마다 값이 하나일 때만, `pivot_table( )`은 여럿이어도 쓸 수 있다는 것이 차이입니다.


**[결과 읽기]** 넓은 표(365 × 24)와 깔끔한 표(8,760 × 3)는 같은 값을 담은 두 모양이고, 오가도 값이 하나도 바뀌지 않았습니다. 넓은 표 그림에서는 출퇴근 띠와 계절, 운영 중단 날이 한 장에 보입니다. 요약표를 보면 어느 계절이든 평일이 쉬는 날보다 한 시간 평균이 많습니다. 미니 프로젝트의 지하철 파일이 바로 넓은 표이고, 성적표("이름, 국어, 영어, 수학")도 `melt(id_vars="이름", var_name="과목", value_name="점수")` 한 줄로 깔끔한 표가 됩니다([연습문제 3-1]).

#### [실습 예제 3-13 ☆] 계산한 습도로 채우고 표시 남기기

습도 결측 17시간을 [실습 예제 3-6]에서 계산한 값으로 채우고, 채웠다는 표시 열을 만드시오. 채운 날의 하루 습도를 그려 앞뒤와 자연스럽게 이어지는지 확인하시오.

이 실습에서 쓰는 것: 표시 열, `fillna(시리즈)`, `groupby( ).transform( )`

**❶ 표시 열을 먼저 만들고 채우기**

In [ ]:
df["humidity_calc"] = df["humidity"].isna()             # 채웠다는 표시
df["humidity"] = df["humidity"].fillna(calc.round())
print(df.loc[df["humidity_calc"], ["date", "hour", "temp", "dew", "humidity"]].head(3).to_string(index=False))
print("채운 시간:", df["humidity_calc"].sum(), " 남은 빈칸:", df["humidity"].isna().sum())

- 순서가 중요합니다. 채우기 **전에** `isna( )`로 표시 열을 만들어야 합니다. 채운 뒤에는 빈칸이 없어 어디를 채웠는지 알 수 없습니다.
- `fillna(calc.round( ))`: 값 하나가 아니라 시리즈를 주면, 빈칸마다 **같은 행**의 calc 값으로 채웁니다. 행마다 그 시간의 기온·이슬점으로 계산한 습도가 들어갑니다.
- 남은 빈칸이 0인지 확인합니다.


**❷ 채운 날을 그래프로 확인하기**

In [ ]:
d = df[df["date"] == "2018-05-21"]
plt.figure(figsize=(6, 3))
plt.plot(d["hour"], d["humidity"], color="#3F7CC0", marker="o", markersize=3, label="습도")
plt.scatter(d.loc[d["humidity_calc"], "hour"], d.loc[d["humidity_calc"], "humidity"],
            color="#E04B3C", s=40, zorder=3, label="계산으로 채운 시간")
plt.xlabel("시각"); plt.ylabel("습도(%)"); plt.legend(); plt.show()
print("이날 채운 시각:", d.loc[d["humidity_calc"], "hour"].tolist())

- `df["date"] == "2018-05-19"`: 날짜형 열은 "연-월-일" 글자와 바로 견줄 수 있습니다. 채운 시간이 가장 많은(6시간) 하루를 고릅니다.
- `marker="o"`: 선 위의 시각마다 작은 동그라미를 찍습니다. 채운 시각은 빨간 점으로 덧그립니다. `zorder=3`은 점을 선 위에 올려 그리라는 뜻입니다.


*위 그래프: 2018년 5월 21일의 시간별 습도. 계산으로 채운 시각(빨강)이 앞뒤 시각과 자연스럽게 이어진다.*

**[직접 바꿔 보기]** 계산식이 없다면 "같은 시각의 중앙값"으로 채울 수도 있습니다. 두 방법을 견주시오.

In [ ]:
by_hour = df["humidity"].where(~df["humidity_calc"]).groupby(df["hour"]).transform("median")
gap = (by_hour - df["humidity"])[df["humidity_calc"]].abs()
print("같은 시각 중앙값으로 채웠다면, 계산값과의 차이 평균:", round(gap.mean(), 1), "%p")

`where(~표시 열)`로 채운 값을 잠시 다시 NaN으로 돌리고, `groupby("hour").transform("median")`으로 행마다 같은 시각의 중앙값을 붙였습니다. 이 값은 그날의 날씨를 모르므로 계산값과 크게 어긋납니다. 증인 열이 있으면 증인으로, 없으면 집단의 대푯값으로 채우되, 어느 쪽이든 표시 열을 남깁니다.

**[결과 읽기]** humidity_calc 열이 True인 17시간은 잰 값이 아니라 계산한 값이라는 표시입니다. 그래프에서도 채운 시각이 앞뒤 시각과 자연스럽게 이어집니다. 나중에 이 열로 채운 시간만 빼고 분석하거나, "채운 값이 결과를 바꾸는지" 견주어 볼 수 있습니다.

> **핵심 3줄**
>
> 인코딩, 날짜 형식, 조건 필터, IQR 규칙, `groupby( )`, `merge( )`를 차례로 써서 정제된 하루 단위 표를 만들었습니다.
>
> 판다스 함수는 새 표를 돌려주므로 받아 두고, 단계마다 행 수와 기간을 출력하고 그래프로 확인합니다.
>
> 결과는 bike_daily_clean.csv(352일)와 bike_hourly_clean.csv(8,465시간)로 저장했고, 원본 raw는 고치지 않았습니다.

## [미니 프로젝트 3] 지하철 승객은 해마다 얼마나 늘었을까

**질문.** 코로나19가 한창이던 2021년부터 2024년까지, 서울 지하철 2호선 강남역, 신림역, 홍대입구역의 승객은 얼마나 늘었을까요? 답은 공공데이터포털에 있습니다. 그런데 받은 파일을 그대로 더하면 엉뚱한 답이 나옵니다.

**데이터.** 서울교통공사가 공공데이터포털에 올린 "역별 일별 시간대별 승하차인원 정보"입니다. 한 행은 "어느 날, 어느 역의 승차(또는 하차)"이고, 열은 날짜, 호선, 역번호, 역명, 승하차 구분과 시간대별 인원입니다. 시간대가 열 이름인 넓은 표입니다. 강의 자료의 지하철_원자료_표본.zip에는 포털에서 받은 원자료 여섯 파일에서 세 역의 줄만 글자 하나 바꾸지 않고 옮겨 두었습니다(파일마다 720~2,196행). 지저분한 모습도 원자료 그대로입니다. 이 zip 파일을 코랩 왼쪽 파일 창에 끌어다 놓고 시작합니다. 정제를 마친 정답 파일(지하철_3개역_정제_2021_2024.csv)은 강의 자료에 따로 있습니다.

> **[더 알아보기] ☆ 포털에서 원자료를 직접 받기**
>
> 공공데이터포털(data.go.kr)에서 "서울교통공사 역별 일별 시간대별 승하차인원 정보"를 검색하면 해마다의 CSV 파일을 받을 수 있습니다. 한 해에 약 20만 행, 20~30MB입니다. 2023년은 4월, 9월, 12월 기준의 파일 셋을 받았는데, 기간이 서로 겹쳤습니다. 받은 파일을 subway_raw 폴더에 넣고 1단계의 zip 줄만 지우면, 같은 코드가 모든 역에 그대로 돌아갑니다. 5장의 미니 프로젝트는 이렇게 정제한 2023~2024년 자료를 서울 전체로 더해 씁니다.


**1단계: 데이터 신원 확인**([파이썬 실습 3-1] 응용). zip을 풀고 파일마다 cp949로 읽어, 행·열 수, 기간, 둘째·셋째 열 이름, 마지막 열 이름, 빈칸 수를 출력하시오.

In [ ]:
import zipfile, glob
zipfile.ZipFile("지하철_원자료_표본.zip").extractall("subway_raw")   # 강의 자료 파일
raw_s = {}
for f in sorted(glob.glob("subway_raw/*.csv")):
    t = pd.read_csv(f, encoding="cp949")       # 공공데이터포털 파일의 한글은 cp949
    tag = f[-12:-4]                            # 파일 이름 끝의 기준일, 예: 20211231
    raw_s[tag] = t
    print(tag, t.shape, t.iloc[0, 1], "~", t.iloc[-1, 1], list(t.columns[1:3]), t.columns[-1],
          "빈칸", t.isna().sum().sum())

파일마다 모양이 다릅니다. 2021년 파일은 날짜 열 이름이 "날짜"이고 맨 끝에 "합 계" 칸이 있습니다. 2022년부터는 "수송일자"이고 합계가 없습니다. 2023년은 1~4월, 1~9월, 1~12월 파일이 겹쳐 있습니다. 2022년 파일에는 빈칸이 906개 있습니다. 1~5월의 "24시이후" 칸이 모두 비어 있습니다.

**2단계: 기준선 만들기.** 정제하지 않고, 파일마다 숫자 열을 모두 더해 해마다 하루 평균 승객을 구하시오. 연번과 역번호 열만 빼시오.

In [ ]:
naive = {}
for tag, t in raw_s.items():                   # 정제하지 않고: 숫자 열을 모두 더한다
    num = t.select_dtypes("number").drop(columns=["연번", "역번호"], errors="ignore")
    naive[tag[:4]] = naive.get(tag[:4], 0) + num.sum().sum()
days = {"2021": 365, "2022": 365, "2023": 365, "2024": 366}
naive = pd.Series({y: naive[y] / days[y] for y in days}).round()
print("정제 전 하루 평균(세 역):", naive.astype(int).to_dict())

정제 전 숫자는 2021년 66만 명에서 2022년 37만 명으로 반 토막이 났다가, 2023년 81만 명으로 두 배가 됩니다. 말이 되지 않습니다. 이 들쭉날쭉한 숫자가 정제로 고쳐야 할 출발점입니다.

**3단계: 열 맞추고 잇고 겹침 지우기**([파이썬 실습 3-6], [3-9] 응용). 열 이름과 시간대 칸을 해마다 같게 맞추어 잇고, 같은 날짜·역·구분이 겹치면 가장 나중 파일을 남기시오.

In [ ]:
hours = ["06시이전"] + [f"{k:02d}-{k + 1:02d}시간대" for k in range(6, 23)] + ["23시이후"]
parts = []
for tag, t in raw_s.items():
    t = t.copy()
    if "합 계" in t.columns:                       # 2021년: 시간대 19칸 + 합계
        t = t.drop(columns="합 계")
    else:                                          # 2022년부터: 20칸 → 23시 이후를 한 칸으로
        t["23시이후"] = t["23-24시간대"] + t["24시이후"].fillna(0)   # 빈칸은 0으로 본다
        t = t.drop(columns=["23-24시간대", "24시이후"])
    t.columns = ["연번", "날짜", "호선", "역번호", "역명", "구분"] + hours
    t["호선"] = t["호선"].astype(str).str.replace("호선", "") + "호선"   # 2 → 2호선
    t["파일"] = tag
    parts.append(t)
sub = pd.concat(parts, ignore_index=True)
sub["날짜"] = pd.to_datetime(sub["날짜"])
print("이어 붙인 행:", len(sub))
key = ["날짜", "역명", "구분"]
print("같은 날짜·역·구분이 겹친 행:", sub.duplicated(key, keep=False).sum())
q3_23 = sub[(sub["날짜"] >= "2023-07-01") & (sub["날짜"] <= "2023-09-30")]
print("2023년 7~9월 합계:", q3_23.groupby("파일")[hours].sum().sum(axis=1).to_dict())
sub = sub.sort_values(["파일"] + key).drop_duplicates(key, keep="last")   # 가장 나중 파일을 믿는다
print("겹침을 지운 뒤:", len(sub), "행,", sub["날짜"].nunique(), "일")

2021년의 "23시 이후"는 한 칸이고, 2022년부터는 "23-24시간대"와 "24시이후" 두 칸입니다. 그래서 2022년부터의 두 칸을 더해 한 칸으로 맞추었습니다. 2022년 1~5월의 빈칸은 0으로 보았습니다. 같은 날짜·역·구분이 겹친 행은 3,996행이었습니다. 겹친 행의 값이 늘 같지는 않았습니다. 2023년 7~9월 합계가 9월 파일은 34,413,858명, 12월 파일은 35,351,557명이었습니다. 나중 파일에서 값이 고쳐진 것으로 보고, 가장 나중 파일을 믿었습니다. 겹침을 지우자 1,461일 × 6행 = 8,766행이 남았습니다.

**4단계: 이상한 값 찾고 결과 견주기**([파이썬 실습 3-4], [3-10] 응용). 달마다 "13~14시 ÷ 12~13시"를 구해 튀는 달을 찾으시오. 그 칸을 앞뒤 시간대의 평균으로 채우고 표시 열을 남긴 뒤, 해마다 하루 평균 승객을 정제 전과 견주시오.

In [ ]:
sub["달"] = sub["날짜"].dt.strftime("%Y-%m")
ratio = (sub.groupby("달")["13-14시간대"].sum() / sub.groupby("달")["12-13시간대"].sum()).round(2)
print("13~14시 ÷ 12~13시가 가장 큰 달:", ratio.sort_values().tail(3).to_dict())
bad = sub["달"] == "2024-04"
sub["채움"] = bad                                   # 고친 칸이 있는 행 표시
sub.loc[bad, "13-14시간대"] = ((sub.loc[bad, "12-13시간대"] + sub.loc[bad, "14-15시간대"]) / 2).round()
sub["합계"] = sub[hours].sum(axis=1)
clean = sub.groupby(sub["날짜"].dt.year)["합계"].sum() / sub.groupby(sub["날짜"].dt.year)["날짜"].nunique()
clean.index = clean.index.astype(str)
result = pd.DataFrame({"정제 전": naive, "정제 후": clean.round()})
print(result.astype(int).to_string())
cols = ["날짜", "호선", "역번호", "역명", "구분"] + hours + ["합계", "채움", "파일"]
sub[cols].to_csv("지하철_3개역_정제_2021_2024.csv", index=False, encoding="utf-8-sig")

(result / 10000).plot.bar(color=["#B9C0C8", "#E04B3C"], figsize=(6, 3))
plt.ylabel("하루 평균 승하차(만 명)")
plt.xlabel("")
plt.xticks(rotation=0)
plt.show()

보통 13~14시 승객은 12~13시와 비슷합니다(1.0~1.1배). 그런데 2024년 4월만 7.59배였습니다. 신림역 승차는 4월 13~14시 평균이 33,144명으로, 다른 달(약 2,600명)의 열두 배가 넘습니다. 이 값은 포털에 올라온 원자료에 그대로 있습니다. 데이터 과학자 스키나는 기록이 잘못된 오류(error)와, 자료를 만들고 옮기는 과정이 남긴 흔적(artifact)을 구별하라고 합니다. 이 칸은 오류로 보고 채웠습니다. 5장의 미니 프로젝트는 같은 오류가 있는 2024년 4월을 통째로 빼기로 합니다. 어느 쪽이든 일지에 남기면 됩니다. 2022년 1~5월의 빈칸은 흔적일 수도 있습니다. 그때 자정 넘어 다닌 열차가 없었는지, 집계가 빠졌는지는 파일만으로 알 수 없습니다.

정제한 뒤 세 역의 하루 평균 승객은 2021년 329,944명에서 2024년 406,172명으로 해마다 꾸준히 늘었습니다. 정제 전의 "반 토막, 두 배"는 합계 칸을 한 번 더 더하고(2021년), 겹친 파일을 모두 더한(2023년) 탓이었습니다.


**5단계: 한계 쓰기.** 이 정제가 아직 모르는 것을 [표 3-6]처럼 세 가지 이상 쓰시오. 2024년 4월 13~14시를 채운 방법과, 2022년 빈칸을 0으로 본 결정은 반드시 다루어야 합니다.

**제출물.** 그림 2장(정제 전후의 해마다 하루 평균, 13~14시 ÷ 12~13시의 달별 변화), 정제 일지 1개([표 3-5] 형식), 결론 5문장 이내.

**[표 3-8] 미니 프로젝트 채점 기준**

| 항목(비중) | 상 | 중 | 하 |
|---|---|---|---|
| 데이터 점검(20%) | 인코딩, 열 이름, 겹친 기간, 빈칸을 파일마다 밝힘 | 일부만 확인함 | 점검 없이 바로 더함 |
| 기준선 비교(20%) | 정제 전 숫자를 구하고 무엇이 그 숫자를 비틀었는지 설명함 | 정제 전 숫자만 구함 | 정제 전후를 견주지 않음 |
| 기법 적용(30%) | 열 맞추기, 겹침 지우기(나중 파일), 이상한 칸 처리와 표시 열이 모두 맞음 | 하나가 빠지거나 틀림 | 코드가 돌지 않거나 결과가 틀림 |
| 해석과 한계(30%) | 결정마다 "왜"를 일지에 쓰고, 한계를 셋 이상 씀 | 한계를 한두 가지 씀 | 숫자만 나열함 |

**교수용 정답 예시.** 정제 후 세 역의 하루 평균 승하차는 2021년 329,944명, 2022년 368,691명, 2023년 392,876명, 2024년 406,172명으로, 2021년보다 2024년이 23.1% 많습니다. 정제 전 숫자는 659,901명, 370,085명, 812,919명, 415,888명입니다. 2021년은 "합 계" 칸이 시간대 칸과 함께 더해져 두 배가 되었고, 2023년은 겹친 파일 세 개가 모두 더해졌습니다. 2024년의 차이는 4월 13~14시 오류 때문입니다. 겹친 3,996행 가운데 2023년 7~9월 값은 9월 파일과 12월 파일이 달랐고(12월 파일이 2.7% 많음), 학생이 9월 파일을 남겼다면 2023년 숫자가 조금 작게 나옵니다. 전체 풀이는 교수용 노트북에 있습니다.

## ✏️ [더 풀어 보기] 아침 8시끼리의 울타리

[파이썬 실습 3-4]의 `h`에서 아침 8시만 골라 Q1, Q3, 위 울타리를 구하시오. 전체에 쓴 울타리(2,389대)와 견주고, 전체 기준으로 후보였던 8시가 8시 기준으로도 후보인지 확인하시오.

In [ ]:
# 여기에 직접 코드를 써 보십시오.

### 정답 코드

직접 해 본 뒤에 실행해 보십시오.

In [ ]:
eight = h[h["hour"] == 8]["count"]                    # 아침 8시만
a, b = eight.quantile([0.25, 0.75])
print(f"아침 8시 {len(eight)}시간: Q1 {a:.0f}, Q3 {b:,.0f}, 위 울타리 {b + 1.5 * (b - a):,.1f}")
print("전체 기준으로 후보였던 8시:", (eight > upper).sum(), "시간,  8시 기준 후보:", (eight > b + 1.5 * (b - a)).sum(), "시간")